# 07 — Incremental evaluation, fixed cohorts, and configuration comparison

Run after new 06. Evaluates Sales, Rental, and both Eilers indices against independently loaded actuals. Each `(forecast run, fixed cohort, evaluation policy/version)` completes once; completed units are verified/skipped across sessions. Adding candidates refreshes the ranking from saved metrics without rescoring old candidates.

## Setup and normal use

1. Use the shared `configs/experiments.yaml`, or set its absolute `/Workspace/...` path in cell 2. Merge the template's new `evaluation:` section into your existing YAML if you added custom experiments.
2. Use the compatible 05/06 copies supplied here: their YAML loaders accept `evaluation`, without changing their generation/forecast identities or numerical behavior.
3. **No further reset is required if old outputs were cleared earlier.** If legacy 07 outputs remain, use reset scope `07` once to preserve new 05/06 results. Normal batches and actuals refreshes never require a reset.
4. Run this notebook in order, with one active manual writer. Run `Review_Forecasts` separately for experiment/game charts. There is no scheduling or widget-driven evaluation configuration.

## Ground truth and inference

The benchmark universe comes from an independent pinned AI attributes/mapping snapshot, not candidate predictions. Games without actuals remain visible in eligibility and review. Eligibility is per game, channel and metric:

| Actuals | Evaluation treatment |
| --- | --- |
| No observed actuals/mapping | Forecast-only; excluded from error metrics |
| Fewer than the configured observed months | Forecast-only; insufficient-history reason |
| At least `minimum_observed_months` valid observed months | Score those observed months in the assigned split |
| Missing/inconsistent origin or label dates | Invalid-date reason; no malformed points scored |
| Eligible actuals but missing/nonfinite/misaligned prediction | Remains expected; coverage decreases |

A zero is a valid observation. Filled target months do not count as observed. Sales cumulative actuals may include earlier internal zero-filled gaps; `exclude_cumulative_imputed_history` controls whether those later labels are eligible. No target rental/performance forward fill is used as truth.

One canonical game has one chronological train/validation/test assignment, shared across tasks. Equal release dates stay together. `canonical_game_split` retains that membership; a task without eligible labels has `evaluation_split = forecast_only`. No model is fitted here: train metrics are descriptive, validation selects configuration, test remains held out from selection.

## Explicit calendars and frozen labels

The benchmark prediction date is its pinned release date, with 05's explicit fallback only when release is missing. Actual source calendar months are aligned to that date. Source observations starting before prediction are flagged and excluded by default; malformed calendar/month-index/origin alignment is always excluded. Candidate origin/calendar must agree with the frozen expected point; predictions are never rebased using future actual dates.

`evaluation.data_revision` pins all four actual/mapping table versions once. Reuse that revision for future experiments. Change it explicitly (for example, `labels_v2`) to score newly arrived actuals or corrected mappings. That creates new cohorts/evaluations while preserving forecasts and old results. `forecast.data_revision` independently controls the historical inputs used by 06.

Each compatible definition/population family has one frozen cohort per label revision/policy. Families distinguish source definitions, slot populations and calendar policies; similarity weights, cutoffs, candidate lists, input source vintages and global appended output-table versions do not rebuild a cohort. Games newly introduced outside a frozen cohort remain forecast-only/outside-cohort diagnostics until an explicit revision refresh.

## Demand curves

Each sales and rental evaluation scores the saved neighbor mean (`baseline`), linear (`linear`), quadratic (`poly2`), and logarithmic (`log`) forecasts against observed actuals. The `curve_name` column identifies the series in metrics, validation rankings and MLflow. A missing fit reduces coverage for that curve. A saved extrapolation is scored when its month has an eligible observed actual. Historical runs that only saved the baseline receive baseline metrics. Each Eilers index has one baseline series.

## Metrics and recommendations

MAE, RMSE, bias (`prediction - actual`) and directional accuracy are saved per game, overall and actual-based bracket, separately for train/validation/test and each task/metric/population. Overall errors pool observed game-month pairs. Directional accuracy uses consecutive scored months; flat changes form a separate direction.

Top 20%, middle 60% and bottom 20% brackets use actuals at exactly `bracket_month`, separately per task/population/split. No observed value at that month means `not_available`, with no substitution from another horizon. Equal values share a bracket; small groups may not yield exact proportions.

Selection compares run-and-curve candidates by validation-overall MAE, then RMSE, run ID and curve name. Full expected validation-point/game coverage is required; incomplete candidates keep their measured metrics and reasons but cannot win. Test metrics never choose weights. Separate recommendations are saved per task/population; recommendations do not change 05/06 settings.

## Persistent results and monitoring

- `forecast_error_metrics`: complete per-game/overall/bracket accuracy and coverage.
- `forecast_evaluation_eligibility` and `forecast_train_test_split`: fixed labels' eligibility, reasons, splits and brackets.
- `forecast_evaluation_cohorts` and `forecast_evaluation_cohort_runs`: observed expected points and verified cohort completion.
- `forecast_evaluation_runs` and `forecast_evaluation_attempts`: durable per-forecast work and attempts, with processing/tracking status separated.
- `forecast_selected_configurations` and `forecast_selection_runs`: immutable ranking/recommendation snapshots from saved validation metrics.
- `forecast_actuals_snapshots` and `forecast_evaluation_batches`: independently pinned sources and batch summaries/errors.

The final cell shows statuses, failures, eligible/inference counts, metrics and recommendations. A zero-label cohort can complete with no accuracy rows; it is not invented ground truth. Failed units resume with the same IDs and source pins. Independent failures do not discard successful units.

MLflow under `/Shared/AI_Feature_Extraction_Evaluation` records per-forecast evaluations. Tracking outages are retryable and do not downgrade verified Delta completion. All raw-source preparation stays in Spark; only narrow target/task labels and forecast curves enter the pandas metric engine. Local behavioral tests and compilation pass; Spark/Delta/MLflow runtime validation still requires Databricks.

## Training bias correction

For named experiments saved by 06 with `train_bias_correction: true`, 07 learns an offset for each demand curve, revenue type, and relative month. It averages actual minus prediction on aligned training observations dated strictly before the earliest validation or test prediction month. The saved offsets apply to predictions issued from that month onward. Corrected demand is floored at zero. A missing prediction or missing training offset produces a missing corrected prediction. The additional curves are scored on validation and test and are available in Review. Offsets, training game counts, and the cutoff are saved in the evaluation run diagnostics.


In [0]:
# Only notebook input. Evaluation settings come from the shared YAML.
CONFIG_PATH = "configs/experiments.yaml"


In [0]:
path1='development_042_silver_sandbox.demand_forecast.forecast_error_metrics'
p1=spark.table(path1)
display(p1)

## Shared configuration and evaluation identities

Definitions only; these cells do not generate forecasts or change data.


In [0]:
"""Validated YAML configuration and semantic work identities for notebook 05."""
import copy
import hashlib
import json
import math
import re
from datetime import date, datetime
from numbers import Integral, Real
from pathlib import Path

SIMILARITY_IMPLEMENTATION_VERSION = "05_batch_yaml_v1_2026_11_19"
WEIGHT_GROUPS = ("mechanics", "themes", "reels", "persistence", "ways", "months_since_release")
PARENT_PROCESSES = {"p1_parent_games_only", "p2_current", "p3_include_parent"}


def config_json(value):
    return json.dumps(value, sort_keys=True, separators=(",", ":"), allow_nan=False,
                      default=lambda x: x.isoformat() if isinstance(x, (date, datetime)) else str(x))


def config_sha256(value):
    return hashlib.sha256(config_json(value).encode("utf-8")).hexdigest()


def quoted_table(name):
    return ".".join("`" + p.replace("`", "``") + "`" for p in name.split("."))


def _keys(value, allowed, where, required=()):
    if not isinstance(value, dict):
        raise ValueError(f"{where} must be a mapping.")
    unknown, missing = set(value) - set(allowed), set(required) - set(value)
    if unknown or missing:
        raise ValueError(f"{where}: unknown keys={sorted(unknown)}, missing keys={sorted(missing)}")


def _text(value, where):
    if not isinstance(value, str) or not value.strip():
        raise ValueError(f"{where} must be a nonempty string.")
    return value.strip()


def _integer(value, where, minimum=0):
    if isinstance(value, bool) or not isinstance(value, Integral) or value < minimum:
        raise ValueError(f"{where} must be an integer >= {minimum}.")
    return int(value)


def _date(value, where):
    if value is None:
        return None
    if isinstance(value, datetime):
        value = value.date()
    if isinstance(value, date):
        return value.isoformat()
    try:
        return date.fromisoformat(str(value)).isoformat()
    except ValueError as exc:
        raise ValueError(f"{where} must be YYYY-MM-DD or null.") from exc


def _table(value, where):
    value = _text(value, where)
    if len(value.split(".")) != 3 or not all(re.fullmatch(r"[A-Za-z_][A-Za-z0-9_]*", x) for x in value.split(".")):
        raise ValueError(f"{where} must be a three-part catalog.schema.table name.")
    return value


def validate_similarity_settings(settings):
    allowed = {"raw_weights", "distance_method", "parent_game_process", "neighbor_limits",
               "neighbor_released_after", "minimum_age_gap_months"}
    _keys(settings, allowed, "similarity settings", allowed)
    settings = copy.deepcopy(settings)
    settings["distance_method"] = _text(settings["distance_method"], "distance_method").lower()
    if settings["distance_method"] not in {"cosine", "euclidean"}:
        raise ValueError("distance_method must be cosine or euclidean.")
    if settings["parent_game_process"] not in PARENT_PROCESSES:
        raise ValueError(f"Unknown parent_game_process: {settings['parent_game_process']!r}")
    _keys(settings["raw_weights"], WEIGHT_GROUPS, "raw_weights", WEIGHT_GROUPS)
    for key, value in settings["raw_weights"].items():
        if isinstance(value, bool) or not isinstance(value, Real) or not math.isfinite(float(value)) or value < 0:
            raise ValueError(f"raw_weights.{key} must be a finite nonnegative number.")
        settings["raw_weights"][key] = float(value)
    if not any(settings["raw_weights"].values()):
        raise ValueError("At least one feature-group weight must be positive.")
    _keys(settings["neighbor_limits"], {"sales", "rental", "eilers"}, "neighbor_limits", {"sales", "rental", "eilers"})
    settings["neighbor_limits"] = {k: _integer(v, f"neighbor_limits.{k}") for k, v in settings["neighbor_limits"].items()}
    settings["minimum_age_gap_months"] = _integer(settings["minimum_age_gap_months"], "minimum_age_gap_months")
    settings["neighbor_released_after"] = _date(settings["neighbor_released_after"], "neighbor_released_after")
    if settings["neighbor_released_after"] is None:
        raise ValueError("neighbor_released_after must be a date.")
    return settings


def named_forecast_settings(item, similarity):
    """Weights are ordered by historical demand in each month, highest first."""
    name = item["name"]
    spec = item.get("forecast", {})
    _keys(spec, {"rank_weights", "train_bias_correction"}, f"{name}.forecast")
    weights = spec.get("rank_weights", [1.0])
    if not isinstance(weights, list) or not weights:
        raise ValueError(f"{name}.forecast.rank_weights must be a nonempty list.")
    if any(isinstance(w, bool) or not isinstance(w, Real) or not math.isfinite(w) or w <= 0 for w in weights):
        raise ValueError(f"{name}.forecast.rank_weights must contain positive finite numbers.")
    if any(a < b for a, b in zip(weights, weights[1:])):
        raise ValueError(f"{name}.forecast.rank_weights must be ordered highest weight first.")
    if not math.isfinite(sum(weights)):
        raise ValueError(f"{name}.forecast.rank_weights sum must be finite.")
    correct = spec.get("train_bias_correction", False)
    if not isinstance(correct, bool):
        raise ValueError(f"{name}.forecast.train_bias_correction must be true or false.")
    return {"name": name, "rank_weights": [float(w) for w in weights],
            "train_bias_correction": correct, "similarity_settings_hash": config_sha256(similarity)}


def load_experiment_config(config_path, require_enabled=True):
    try:
        import yaml
    except ImportError as exc:
        raise ImportError("PyYAML is required. Install PyYAML on your Databricks environment, then rerun.") from exc

    # Reject duplicate YAML keys rather than silently replacing a weight/setting.
    class UniqueLoader(yaml.SafeLoader):
        pass
    def unique_mapping(loader, node, deep=False):
        result = {}
        for key_node, value_node in node.value:
            key = loader.construct_object(key_node, deep=deep)
            if key in result:
                raise ValueError(f"Duplicate YAML key {key!r} on line {key_node.start_mark.line + 1}.")
            result[key] = loader.construct_object(value_node, deep=deep)
        return result
    UniqueLoader.add_constructor(yaml.resolver.BaseResolver.DEFAULT_MAPPING_TAG, unique_mapping)
    path = Path(config_path).expanduser().resolve()
    if not path.is_file():
        raise FileNotFoundError(f"Configuration not found: {path}. Put experiments.yaml in configs beside this notebook, or set CONFIG_PATH to its absolute /Workspace/... path.")
    original_text = path.read_text(encoding="utf-8")
    raw = yaml.load(original_text, Loader=UniqueLoader)
    sections = {"schema_version", "data", "outputs", "tracking", "monitoring", "similarity_defaults", "experiments"}
    # Downstream stages share this file without changing 05's semantic inputs.
    _keys(raw, sections | {"forecast", "evaluation"}, "configuration", sections)
    if isinstance(raw["schema_version"], bool) or raw["schema_version"] != 1:
        raise ValueError("Only schema_version: 1 is supported.")
    _keys(raw["data"], {"revision", "ai_attributes_table", "source_version", "prediction_as_of_date"}, "data",
          {"revision", "ai_attributes_table", "source_version", "prediction_as_of_date"})
    data = copy.deepcopy(raw["data"])
    data["revision"] = _text(data["revision"], "data.revision")
    data["ai_attributes_table"] = _table(data["ai_attributes_table"], "data.ai_attributes_table")
    if data["source_version"] is not None:
        data["source_version"] = _integer(data["source_version"], "data.source_version")
    data["prediction_as_of_date"] = _date(data["prediction_as_of_date"], "data.prediction_as_of_date")
    output_keys = {"like_games_table", "runs_table", "attempts_table"}
    _keys(raw["outputs"], output_keys, "outputs", output_keys)
    outputs = {k: _table(v, f"outputs.{k}") for k, v in raw["outputs"].items()}
    if len(set(outputs.values())) != len(outputs) or data["ai_attributes_table"] in outputs.values():
        raise ValueError("Output tables must be distinct and cannot be the AI source table.")
    _keys(raw["tracking"], {"experiment_path"}, "tracking", {"experiment_path"})
    tracking_path = _text(raw["tracking"]["experiment_path"], "tracking.experiment_path")
    if not tracking_path.startswith("/"):
        raise ValueError("tracking.experiment_path must start with /.")
    _keys(raw["monitoring"], {"history_rows"}, "monitoring", {"history_rows"})
    history_rows = _integer(raw["monitoring"]["history_rows"], "monitoring.history_rows", 1)
    defaults = validate_similarity_settings(raw["similarity_defaults"])
    if not isinstance(raw["experiments"], list) or not raw["experiments"]:
        raise ValueError("experiments must be a nonempty list.")
    experiments, names = [], set()
    for index, item in enumerate(raw["experiments"]):
        _keys(item, {"name", "enabled", "overrides", "forecast"}, f"experiments[{index}]", {"name"})
        name = _text(item["name"], f"experiments[{index}].name")
        if name in names:
            raise ValueError(f"Duplicate experiment name: {name!r}")
        names.add(name)
        enabled = item.get("enabled", True)
        if not isinstance(enabled, bool):
            raise ValueError(f"{name}.enabled must be true or false.")
        overrides = item.get("overrides", {})
        _keys(overrides, defaults, f"{name}.overrides")
        merged = copy.deepcopy(defaults)
        for key, value in overrides.items():
            if key in {"raw_weights", "neighbor_limits"}:
                _keys(value, defaults[key], f"{name}.overrides.{key}")
                merged[key].update(copy.deepcopy(value))
            else:
                merged[key] = copy.deepcopy(value)
        merged = validate_similarity_settings(merged)
        forecast = named_forecast_settings(item, merged)
        experiments.append({"name": name, "enabled": enabled, "settings": merged, "forecast": forecast})
    if require_enabled and not any(x["enabled"] for x in experiments):
        raise ValueError("Enable at least one experiment in the YAML file.")
    return {"data": data, "outputs": outputs, "tracking": {"experiment_path": tracking_path},
            "monitoring": {"history_rows": history_rows}, "experiments": experiments,
            "config_path": str(path), "config_text": original_text,
            "yaml_sha256": hashlib.sha256(original_text.encode("utf-8")).hexdigest()}


def similarity_work_identity(experiment, data, source_version):
    semantic = {"implementation_version": SIMILARITY_IMPLEMENTATION_VERSION,
                "data_revision": data["revision"], "source_table": data["ai_attributes_table"],
                "source_table_version": int(source_version),
                "prediction_as_of_date": data["prediction_as_of_date"],
                "similarity": copy.deepcopy(experiment["settings"])}
    digest = config_sha256(semantic)
    return digest, semantic


def preflight_similarity_tables(spark, config):
    """Refuse to mix pre-ledger legacy generations with the new history."""
    results, runs = (config["outputs"][x] for x in ["like_games_table", "runs_table"])
    if spark.catalog.tableExists(results):
        frame = spark.table(results)
        if frame.limit(1).count():
            if "similarity_work_key" not in frame.columns or not spark.catalog.tableExists(runs):
                raise ValueError("Previous similarity results use the old format. Run 00_Reset_Similarity_Pipeline once before the first new batch. No results were overwritten.")
            ledger = spark.table(runs)
            if not {"work_key", "experiment_id", "metadata_json"}.issubset(ledger.columns):
                raise ValueError("The similarity status table has an incompatible schema. Run the one-time reset before starting.")
            if frame.select("experiment_id").distinct().join(ledger.select("experiment_id").distinct(), "experiment_id", "left_anti").limit(1).count():
                raise ValueError("Similarity rows without a matching status record were found. Resolve/reset them before running a new batch.")


def resolve_similarity_source_version(spark, config):
    """A data revision pins one source snapshot across later manually run batches."""
    from pyspark.sql import functions as F
    data, runs = config["data"], config["outputs"]["runs_table"]
    pinned = []
    if spark.catalog.tableExists(runs):
        ledger = spark.table(runs)
        if "metadata_json" not in ledger.columns:
            raise ValueError("Incompatible similarity status table; perform the one-time reset first.")
        pinned = [int(r["source_table_version"]) for r in ledger.where(
            (F.get_json_object("metadata_json", "$.data_revision") == data["revision"])
            & (F.col("source_path") == data["ai_attributes_table"])
        ).select("source_table_version").distinct().collect()]
    if len(set(pinned)) > 1:
        raise ValueError("This data revision has multiple source snapshots. Choose a new revision after resolving the history.")
    if pinned:
        version = pinned[0]
        if data["source_version"] is not None and data["source_version"] != version:
            raise ValueError(f"Revision {data['revision']!r} is already pinned to Delta version {version}. Change data.revision to use another version.")
        return version
    if data["source_version"] is not None:
        return data["source_version"]
    return int(spark.sql(f"DESCRIBE HISTORY {quoted_table(data['ai_attributes_table'])}").agg(F.max("version").alias("version")).first()["version"])


In [0]:
"""Notebook 06 YAML validation, immutable work keys, and source-revision pinning."""
import copy
import hashlib
import json
import math
from numbers import Integral, Real

FORECAST_IMPLEMENTATION_VERSION = "06_batch_yaml_v1_2026_11_19"
FORECAST_SOURCE_KEYS = {"sales", "lease", "performance"}
FORECAST_OUTPUT_KEYS = {"demand", "performance", "neighbors", "runs", "attempts"}
DEFAULT_HISTORY_MONTHS = {"sales_min_observed_months": 6, "rental_min_observed_months": 9}
FORECAST_SETTING_KEYS = {"distance_cutoff", "minimum_neighbors", "horizon_months", "slot_counts",
                         "lease_value_column", "lease_metric_name", "lease_curve_label", "performance_metrics"}


def load_forecast_config(config_path):
    # Shared loader rejects duplicate YAML keys and validates the existing 05
    # sections. 05 ignores this downstream section for work identity purposes.
    common = load_experiment_config(config_path, require_enabled=False)
    import yaml
    raw = yaml.safe_load(common["config_text"])
    forecast = raw.get("forecast")
    forecast_keys = {"data_revision", "sources", "source_versions", "settings", "outputs", "tracking", "selection", "monitoring"}
    if isinstance(forecast, dict) and "experiments" in forecast:
        raise ValueError("Move forecast.experiments into the named entries in the top-level experiments list.")
    _keys(forecast, forecast_keys, "forecast", forecast_keys)
    revision = _text(forecast["data_revision"], "forecast.data_revision")
    _keys(forecast["sources"], FORECAST_SOURCE_KEYS, "forecast.sources", FORECAST_SOURCE_KEYS)
    sources = {k: _table(v, f"forecast.sources.{k}") for k, v in forecast["sources"].items()}
    _keys(forecast["source_versions"], FORECAST_SOURCE_KEYS, "forecast.source_versions", FORECAST_SOURCE_KEYS)
    versions = {k: None if v is None else _integer(v, f"forecast.source_versions.{k}")
                for k, v in forecast["source_versions"].items()}
    _keys(forecast["outputs"], FORECAST_OUTPUT_KEYS, "forecast.outputs", FORECAST_OUTPUT_KEYS)
    outputs = {k: _table(v, f"forecast.outputs.{k}") for k, v in forecast["outputs"].items()}
    protected_inputs = {common["data"]["ai_attributes_table"], *sources.values(), *common["outputs"].values()}
    if len(set(outputs.values())) != len(outputs) or set(outputs.values()) & protected_inputs:
        raise ValueError("Forecast output tables must be distinct and cannot replace any source or similarity tables.")
    _keys(forecast["settings"], FORECAST_SETTING_KEYS | set(DEFAULT_HISTORY_MONTHS), "forecast.settings", FORECAST_SETTING_KEYS)
    settings = copy.deepcopy(forecast["settings"])
    for key, default in DEFAULT_HISTORY_MONTHS.items():
        settings[key] = _integer(settings.get(key, default), key, 1)
    cutoff = settings["distance_cutoff"]
    if isinstance(cutoff, bool) or not isinstance(cutoff, Real) or not math.isfinite(float(cutoff)) or cutoff < 0:
        raise ValueError("forecast.settings.distance_cutoff must be finite and >= 0.")
    settings["distance_cutoff"] = float(cutoff)
    settings["minimum_neighbors"] = _integer(settings["minimum_neighbors"], "minimum_neighbors")
    settings["horizon_months"] = _integer(settings["horizon_months"], "horizon_months", 1)
    slots = settings["slot_counts"]
    if not isinstance(slots, list) or any(isinstance(x, bool) or not isinstance(x, Real) or not math.isfinite(float(x)) or x <= 0 for x in slots):
        raise ValueError("slot_counts must be a list of finite positive numbers; [] means all positive counts.")
    if len(set(slots)) != len(slots):
        raise ValueError("slot_counts cannot contain duplicates.")
    settings["slot_counts"] = sorted(float(x) for x in slots)
    for key in ["lease_value_column", "lease_metric_name", "lease_curve_label"]:
        settings[key] = _text(settings[key], key)
    if not isinstance(settings["performance_metrics"], list) or not settings["performance_metrics"]:
        raise ValueError("performance_metrics must be a nonempty list.")
    settings["performance_metrics"] = [_text(x, "performance_metrics entry") for x in settings["performance_metrics"]]
    if len(set(settings["performance_metrics"])) != len(settings["performance_metrics"]):
        raise ValueError("performance_metrics cannot contain duplicates.")
    settings["performance_metrics"] = sorted(settings["performance_metrics"])
    _keys(forecast["tracking"], {"experiment_path"}, "forecast.tracking", {"experiment_path"})
    mlflow_path = _text(forecast["tracking"]["experiment_path"], "forecast.tracking.experiment_path")
    if not mlflow_path.startswith("/"):
        raise ValueError("forecast.tracking.experiment_path must be an absolute workspace path.")
    _keys(forecast["selection"], {"similarity_experiment_ids"}, "forecast.selection", {"similarity_experiment_ids"})
    ids = forecast["selection"]["similarity_experiment_ids"]
    if not isinstance(ids, list):
        raise ValueError("similarity_experiment_ids must be a list; [] discovers all completed 05 experiments.")
    ids = [_text(x, "similarity_experiment_ids entry") for x in ids]
    if len(set(ids)) != len(ids):
        raise ValueError("similarity_experiment_ids cannot contain duplicates.")
    _keys(forecast["monitoring"], {"history_rows"}, "forecast.monitoring", {"history_rows"})
    return {"common": common,
            "data_revision": revision, "sources": sources, "source_versions": versions,
            "settings": settings, "outputs": outputs, "tracking": {"experiment_path": mlflow_path},
            "selection": {"similarity_experiment_ids": ids},
            "monitoring": {"history_rows": _integer(forecast["monitoring"]["history_rows"], "forecast.monitoring.history_rows", 1)}}


def forecast_work_identity(upstream, config, source_versions):
    # Descriptive labels, requested batch membership, MLflow IDs/paths, output
    # locations, and the global appended like-games table version are not inputs.
    settings = {k: copy.deepcopy(v) for k, v in config["settings"].items() if k != "lease_curve_label"}
    semantic = {
        "implementation_version": FORECAST_IMPLEMENTATION_VERSION,
        "forecast_data_revision": config["data_revision"],
        "similarity_experiment_id": upstream["experiment_id"],
        "similarity_work_key": upstream["work_key"],
        "similarity_config_hash": upstream["config_hash"],
        "similarity_result_hash": upstream["result_hash"],
        "source_paths": copy.deepcopy(config["sources"]),
        "source_versions_resolved": {k: int(source_versions[k]) for k in sorted(FORECAST_SOURCE_KEYS)},
        **settings,
        "prediction_date_rule": "saved similarity prediction_date, falling back to target game_release_date",
        "neighbor_observation_rule": "source_month < first_day_of_target_prediction_month",
        "reporting_assumption": "completed prior month available; publication timestamps not supplied",
        "calendar_rule": "target forecast starts at explicit prediction month; neighbor age starts at first historically observed source month",
        "target_actual_policy": "predictions only; actuals loaded independently by versioned evaluation",
        "sales_fill": "internal missing monthly rows zero, then cumulative sum; no trailing fill",
        "lease_fill": "internal missing monthly levels forward-filled; no trailing fill",
        "performance_fill": "internal missing per-metric neighbor statistics forward-filled; leading null remains; no trailing fill",
        "performance_weighting": "positive no_of_slots; per-metric nonnull denominator",
        "neighbor_selection": "saved channel list; deduplicate source names, exclude target mapping, cutoff then nearest minimum fallback",
        "p3_parent_after_cutoff": "normal cutoff/fallback applies to saved parent candidate",
        "bands": "population standard deviation across contributing neighbors; mean +/- 2 std, not calibrated intervals",
    }
    return config_sha256(semantic), semantic


def preflight_forecast_tables(spark, config):
    outputs = config["outputs"]
    for label in ["demand", "performance", "neighbors"]:
        path = outputs[label]
        if spark.catalog.tableExists(path):
            frame = spark.table(path)
            if frame.limit(1).count():
                if "forecast_work_key" not in frame.columns or not spark.catalog.tableExists(outputs["runs"]):
                    raise ValueError("Legacy forecast results were found. Clear only the old 06/07 outputs using reset scope '06_07' before this migration. Preserve your new 05 results.")
                runs = spark.table(outputs["runs"])
                if not {"work_key", "forecast_run_id", "metadata_json"}.issubset(runs.columns):
                    raise ValueError("The forecast status table uses the old schema. Migrate/reset scope '06_07'; preserve 05.")
                if frame.select("forecast_run_id").distinct().join(runs.select("forecast_run_id").distinct(), "forecast_run_id", "left_anti").limit(1).count():
                    raise ValueError("Forecast rows without status records were found. Resolve them before retrying this batch.")


def resolve_forecast_source_versions(spark, config):
    from pyspark.sql import functions as F
    table = config["outputs"]["runs"]
    matches = []
    if spark.catalog.tableExists(table):
        ledger = spark.table(table)
        if "metadata_json" not in ledger.columns:
            raise ValueError("Legacy forecast manifest found. Reset scope '06_07' before using new 06; preserve 05.")
        matches = ledger.where(F.get_json_object("metadata_json", "$.data_revision") == config["data_revision"]).select(
            "source_versions_json", "config_json").distinct().collect()
    versions = None
    for row in matches:
        previous_config = json.loads(row["config_json"])
        if previous_config["source_paths"] != config["sources"]:
            raise ValueError("This forecast data revision is already assigned to other source tables. Choose another forecast.data_revision.")
        candidate = {k: int(v) for k, v in json.loads(row["source_versions_json"]).items()}
        if versions is not None and candidate != versions:
            raise ValueError("This forecast revision contains conflicting source snapshots; resolve its history before running.")
        versions = candidate
    if versions is not None:
        for key, requested in config["source_versions"].items():
            if requested is not None and requested != versions[key]:
                raise ValueError(f"Revision is pinned to {key} version {versions[key]}; change forecast.data_revision to refresh.")
        return versions
    return {key: int(requested) if requested is not None else int(
        spark.sql(f"DESCRIBE HISTORY {quoted_table(config['sources'][key])}").agg(F.max("version").alias("v")).first()["v"])
        for key, requested in config["source_versions"].items()}


In [0]:
"""07 configuration, fixed-label source revisions, and semantic identities."""
import copy
import hashlib
import json
import math
from numbers import Real

EVALUATION_IMPLEMENTATION_VERSION = "07_demand_curve_metrics_v2"
# Saved 06 demand columns. Eilers has one index forecast per metric.
DEMAND_CURVES = {
    "baseline": ("forecast_value", "Neighbor mean"),
    "linear": ("forecast_linear", "Linear"),
    "poly2": ("forecast_poly2", "Quadratic"),
    "log": ("forecast_log", "Logarithmic"),
}
EVALUATION_SOURCE_KEYS = {"sales", "lease", "performance", "mappings"}
EVALUATION_OUTPUT_KEYS = {"cohorts", "cohort_runs", "splits", "eligibility", "metrics", "selected",
                          "selection_runs", "runs", "attempts", "actuals_snapshots", "batches"}
EVALUATION_SETTING_KEYS = {"evaluation_months", "minimum_observed_months", "bracket_month",
                           "validation_fraction", "test_fraction", "exclude_pre_release_curves",
                           "exclude_cumulative_imputed_history", "performance_metrics"}


def load_evaluation_config(config_path):
    forecast = load_forecast_config(config_path)
    common = forecast["common"]
    import yaml
    section = yaml.safe_load(common["config_text"]).get("evaluation")
    keys = {"data_revision", "sources", "source_versions", "settings", "outputs", "tracking", "selection", "monitoring"}
    _keys(section, keys, "evaluation", keys)
    _keys(section["sources"], EVALUATION_SOURCE_KEYS, "evaluation.sources", EVALUATION_SOURCE_KEYS)
    sources = {k: _table(v, f"evaluation.sources.{k}") for k, v in section["sources"].items()}
    _keys(section["source_versions"], EVALUATION_SOURCE_KEYS, "evaluation.source_versions", EVALUATION_SOURCE_KEYS)
    versions = {k: None if v is None else _integer(v, f"evaluation.source_versions.{k}")
                for k, v in section["source_versions"].items()}
    _keys(section["outputs"], EVALUATION_OUTPUT_KEYS, "evaluation.outputs", EVALUATION_OUTPUT_KEYS)
    outputs = {k: _table(v, f"evaluation.outputs.{k}") for k, v in section["outputs"].items()}
    protected = {*common["outputs"].values(), common["data"]["ai_attributes_table"],
                 *forecast["outputs"].values(), *forecast["sources"].values(), *sources.values()}
    if len(set(outputs.values())) != len(outputs) or set(outputs.values()) & protected:
        raise ValueError("Evaluation outputs must be distinct and cannot replace source, 05, or 06 tables.")
    _keys(section["settings"], EVALUATION_SETTING_KEYS, "evaluation.settings", EVALUATION_SETTING_KEYS)
    settings = copy.deepcopy(section["settings"])
    for key in ["evaluation_months", "minimum_observed_months", "bracket_month"]:
        settings[key] = _integer(settings[key], f"evaluation.settings.{key}", 1)
    if settings["bracket_month"] > settings["evaluation_months"] or settings["minimum_observed_months"] > settings["evaluation_months"]:
        raise ValueError("Minimum observed months and bracket month cannot exceed the evaluation horizon.")
    for key in ["validation_fraction", "test_fraction"]:
        value = settings[key]
        if isinstance(value, bool) or not isinstance(value, Real) or not math.isfinite(float(value)) or not 0 < value < 1:
            raise ValueError(f"{key} must be finite and strictly between 0 and 1.")
        settings[key] = float(value)
    if settings["validation_fraction"] + settings["test_fraction"] >= 1:
        raise ValueError("validation_fraction + test_fraction must be < 1.")
    for key in ["exclude_pre_release_curves", "exclude_cumulative_imputed_history"]:
        if not isinstance(settings[key], bool):
            raise ValueError(f"{key} must be true or false.")
    metrics = settings["performance_metrics"]
    if not isinstance(metrics, list) or not metrics:
        raise ValueError("evaluation performance_metrics must be a nonempty list.")
    metrics = [_text(m, "evaluation performance metric") for m in metrics]
    if len(metrics) != len(set(metrics)):
        raise ValueError("Duplicate evaluation performance metrics.")
    settings["performance_metrics"] = sorted(metrics)
    _keys(section["tracking"], {"experiment_path"}, "evaluation.tracking", {"experiment_path"})
    tracking = _text(section["tracking"]["experiment_path"], "evaluation.tracking.experiment_path")
    if not tracking.startswith("/"):
        raise ValueError("evaluation tracking path must be an absolute workspace path.")
    _keys(section["selection"], {"forecast_run_ids"}, "evaluation.selection", {"forecast_run_ids"})
    ids = section["selection"]["forecast_run_ids"]
    if not isinstance(ids, list):
        raise ValueError("evaluation.selection.forecast_run_ids must be a list.")
    ids = [_text(x, "forecast_run_id") for x in ids]
    if len(ids) != len(set(ids)):
        raise ValueError("Duplicate evaluation forecast IDs.")
    _keys(section["monitoring"], {"history_rows"}, "evaluation.monitoring", {"history_rows"})
    return {"common": common, "forecast": forecast,
            "data_revision": _text(section["data_revision"], "evaluation.data_revision"),
            "sources": sources, "source_versions": versions, "settings": settings, "outputs": outputs,
            "tracking": {"experiment_path": tracking}, "selection": {"forecast_run_ids": ids},
            "monitoring": {"history_rows": _integer(section["monitoring"]["history_rows"], "history_rows", 1)}}


def evaluation_policy(config):
    return {"demand_bias_policy": "monthly training residuals when enabled in the saved named forecast",
            **copy.deepcopy(config["settings"]),
            "missing_release_prediction_date": config["common"]["data"]["prediction_as_of_date"],
            "target_registry": "independent pinned AI mapping/attributes snapshot, not forecast availability",
            "actual_calendar": "source calendar months aligned to target prediction month",
            "sales_actuals": "monthly observed sales summed then cumulative; internal missing months flagged",
            "rental_actuals": "observed monthly source quantity; no target forward fill",
            "performance_actuals": "duplicate slot buckets averaged then positive-slot metric-specific weighting; no target fill",
            "split_rule": "canonical game chronological split; equal release dates stay together; freeze with cohort",
            "selection_rule": "validation MAE, RMSE, forecast run ID and curve name; full coverage required",
            "implementation_version": EVALUATION_IMPLEMENTATION_VERSION}


def evaluation_family(configuration, config):
    """Separate incompatible business populations; source vintages may compare."""
    return {"source_paths": {key: configuration["source_paths"][key] for key in ["sales", "lease", "performance"]},
            "slot_counts": sorted(float(x) for x in configuration.get("slot_counts", [])),
            "lease_value_column": configuration.get("lease_value_column", "source_row_count"),
            "lease_metric_name": configuration.get("lease_metric_name", "monthly_rental_records"),
            "performance_metrics": list(config["settings"]["performance_metrics"]),
            "prediction_date_rule": configuration.get("prediction_date_rule"),
            "calendar_rule": configuration.get("calendar_rule"),
            "neighbor_observation_rule": configuration.get("neighbor_observation_rule")}


def evaluation_work_identity(forecast_record, cohort_id, policy_hash):
    semantic = {"implementation_version": EVALUATION_IMPLEMENTATION_VERSION,
                "forecast_run_id": forecast_record["forecast_run_id"],
                "forecast_config_hash": forecast_record["forecast_config_hash"],
                "cohort_id": cohort_id, "policy_hash": policy_hash}
    return config_sha256(semantic), semantic


SNAPSHOT_SCHEMA = "snapshot_id string, data_revision string, source_paths_json string, source_versions_json string, created_at_utc string"
BATCH_SCHEMA = "batch_id string, data_revision string, status string, started_at_utc string, updated_at_utc string, summary_json string, errors_json string"


def evaluation_merge_record(spark, table, schema, key, record):
    import uuid
    name = "evaluation_record_" + uuid.uuid4().hex
    spark.createDataFrame([record], schema=schema).createOrReplaceTempView(name)
    try:
        spark.sql(f"MERGE INTO {quoted_table(table)} target USING `{name}` source ON target.`{key}` = source.`{key}` "
                  "WHEN MATCHED THEN UPDATE SET * WHEN NOT MATCHED THEN INSERT *")
    finally:
        spark.catalog.dropTempView(name)


def resolve_evaluation_source_versions(spark, config):
    """Pin actuals/mappings before cohort preparation; no moving latest labels."""
    from datetime import datetime, timezone
    from pyspark.sql import functions as F
    path = config["outputs"]["actuals_snapshots"]
    spark.sql(f"CREATE TABLE IF NOT EXISTS {quoted_table(path)} ({SNAPSHOT_SCHEMA}) USING DELTA")
    snapshot_id = config_sha256({"revision": config["data_revision"], "sources": config["sources"]})
    existing = spark.table(path).where(F.col("data_revision") == config["data_revision"]).limit(2).collect()
    if len(existing) > 1:
        raise ValueError("Multiple actuals snapshots own this label revision; resolve before evaluation.")
    if existing:
        record = existing[0].asDict()
        if json.loads(record["source_paths_json"]) != config["sources"]:
            raise ValueError("This evaluation revision is assigned to different source tables; change evaluation.data_revision.")
        versions = {k: int(v) for k, v in json.loads(record["source_versions_json"]).items()}
        if any(v is not None and v != versions[k] for k, v in config["source_versions"].items()):
            raise ValueError("The label revision is already pinned. Change evaluation.data_revision to refresh source versions.")
        return versions
    versions = {key: int(requested) if requested is not None else int(
        spark.sql(f"DESCRIBE HISTORY {quoted_table(config['sources'][key])}").agg(F.max("version").alias("v")).first()["v"])
        for key, requested in config["source_versions"].items()}
    evaluation_merge_record(spark, path, SNAPSHOT_SCHEMA, "snapshot_id", {
        "snapshot_id": snapshot_id, "data_revision": config["data_revision"],
        "source_paths_json": config_json(config["sources"]), "source_versions_json": config_json(versions),
        "created_at_utc": datetime.now(timezone.utc).isoformat()})
    return versions


def preflight_evaluation_tables(spark, config):
    outputs = config["outputs"]
    for label, required in [("runs", {"work_key", "evaluation_id", "metadata_json"}),
                            ("cohorts", {"cohort_id", "target_game_key"}),
                            ("metrics", {"evaluation_id", "policy_hash"})]:
        path = outputs[label]
        if spark.catalog.tableExists(path):
            frame = spark.table(path)
            if frame.limit(1).count() and not required.issubset(frame.columns):
                raise ValueError("Legacy 07 outputs were found. Use reset scope '07' once to preserve new 05/06 results.")


def apply_training_bias(frame, calibration, curve, raw_prediction, origin_column="prediction_date"):
    """Apply the saved additive offset only to forecasts issued after training closed."""
    import numpy as np
    import pandas as pd
    result = pd.Series(np.nan, index=frame.index, dtype=float)
    cutoff = calibration.get("available_from")
    records = calibration.get("offsets", {}).get(curve, [])
    if not cutoff or not records or frame.empty:
        return result
    offsets = pd.DataFrame(records).set_index(["revenue_type", "month_index"])["offset"]
    keys = pd.MultiIndex.from_frame(frame[["revenue_type", "month_index"]])
    adjusted = pd.to_numeric(raw_prediction, errors="coerce") + offsets.reindex(keys).to_numpy()
    available = pd.to_datetime(frame[origin_column], errors="coerce").ge(pd.Timestamp(cutoff))
    return adjusted.clip(lower=0).where(available & np.isfinite(adjusted))


def with_curve_name(frame):
    """Earlier 07 evaluations contain only the raw 06 baseline."""
    from pyspark.sql import functions as F
    curve = F.coalesce("curve_name", F.lit("baseline")) if "curve_name" in frame.columns else F.lit("baseline")
    return frame.withColumn("curve_name", curve)


## Independent actuals and fixed-cohort metrics


In [0]:
"""Independent, versioned evaluation labels and long prediction projections.

All raw-source work remains in Spark. Labels use the saved prediction calendar,
never a target's later first-observation date. Mapping and label availability do
not restrict which targets receive predictions in notebook 06.
"""
from functools import reduce
from numbers import Integral


_EA_DEFAULT_METRICS = ["avg_coin_in_index_vs_house", "avg_theo_net_win_index_vs_house"]
_EA_TASK_KEYS = ["forecast_type", "metric_name", "revenue_type", "own_status"]
_EA_TARGET_COLUMNS = ["target_game_key", "_target_id", "target_game", "game_release_date", "prediction_date"]
_EA_MAPPING_COLUMNS = ["to_sales_name_match", "to_rental_name_match", "to_eilers_ts_name_match"]


def _ea_functions():
    from pyspark.sql import functions as F
    return F


def _ea_require(frame, columns, label):
    missing = sorted(set(columns) - set(frame.columns))
    if missing:
        raise ValueError(f"{label} is missing columns: {missing}")


def _ea_normalized(column):
    F = _ea_functions()
    column = F.col(column) if isinstance(column, str) else column
    return F.regexp_replace(F.lower(F.trim(column.cast("string"))), r"\s+", " ")


def _ea_valid_mapping(column):
    value = _ea_normalized(column)
    return value.isNotNull() & ~value.isin("", "none", "null", "nan", "na", "n/a", "<na>", "__unknown__")


def _ea_finite(column):
    F = _ea_functions()
    # Raw source columns may be string-valued. try_cast protects ANSI runtimes
    # from malformed values; computed expressions below are already numeric.
    value = (F.expr("try_cast(`" + column.replace("`", "``") + "` AS DOUBLE)")
             if isinstance(column, str) else column.cast("double"))
    return F.when(value.isNotNull() & ~F.isnan(value) & (F.abs(value) != float("inf")), value)


def _ea_date(column_name):
    """Malformed source date strings are missing labels under ANSI runtimes."""
    F = _ea_functions()
    return F.expr("try_cast(cast(`" + column_name.replace("`", "``") + "` AS STRING) AS DATE)")


def _ea_metrics(metrics):
    if isinstance(metrics, dict):
        metrics = metrics.get("performance_metrics", _EA_DEFAULT_METRICS)
    result = list(metrics)
    if not result or any(not isinstance(value, str) or not value.strip() for value in result) or len(set(result)) != len(result):
        raise ValueError("Evaluation performance metrics must be distinct nonempty column names.")
    return result


def targets_to_tasks(target_registry_df, metrics, lease_metric_name="monthly_rental_records"):
    """Project every target into sales, rental, and both Eilers populations.

    Target registry must contain target_game_key (join_game_name is accepted as
    a fallback). Canonical name, release/prediction dates and source mappings are
    retained when present. Missing mappings remain visible task records. Pass
    the forecast's lease_metric_name when it differs from the supplied default.
    """
    F = _ea_functions()
    metrics = _ea_metrics(metrics)
    if not isinstance(lease_metric_name, str) or not lease_metric_name.strip():
        raise ValueError("lease_metric_name must be a nonempty string.")
    source = target_registry_df
    if "target_game_key" not in source.columns:
        _ea_require(source, ["join_game_name"], "Evaluation target registry")
        source = source.withColumn("target_game_key", _ea_normalized("join_game_name"))
    else:
        source = source.withColumn("target_game_key", _ea_normalized("target_game_key"))
    if "target_game" not in source.columns:
        source = source.withColumn("target_game", F.col("join_game_name") if "join_game_name" in source.columns else F.col("target_game_key"))
    if "_target_id" not in source.columns:
        source = source.withColumn("_target_id", F.xxhash64("target_game_key"))
    if "game_release_date" not in source.columns:
        source = source.withColumn("game_release_date", F.lit(None).cast("date"))
    else:
        source = source.withColumn("game_release_date", _ea_date("game_release_date"))
    if "prediction_date" not in source.columns:
        source = source.withColumn("prediction_date", F.col("game_release_date"))
    else:
        source = source.withColumn("prediction_date", F.coalesce(_ea_date("prediction_date"), F.col("game_release_date")))
    for name in _EA_MAPPING_COLUMNS:
        if name not in source.columns:
            source = source.withColumn(name, F.lit(None).cast("string"))
    base = source.select(*_EA_TARGET_COLUMNS, *_EA_MAPPING_COLUMNS).where(_ea_valid_mapping("target_game_key")).distinct()
    if base.groupBy("target_game_key").count().where(F.col("count") > 1).limit(1).count():
        raise ValueError("Evaluation target registry has conflicting dates, names, IDs, or mappings for a canonical game. Freeze one canonical target registry per cohort.")

    def project(forecast_type, metric, revenue, ownership, mapping_column):
        return (base.withColumn("forecast_type", F.lit(forecast_type))
                .withColumn("metric_name", F.lit(metric))
                .withColumn("revenue_type", F.lit(revenue))
                .withColumn("own_status", F.lit(ownership))
                .withColumn("mapping_available", _ea_valid_mapping(mapping_column))
                .withColumn("actual_source_key", F.when(_ea_valid_mapping(mapping_column), _ea_normalized(mapping_column)))
                .withColumn("actual_mapping_status", F.when(F.col("mapping_available"), F.lit("mapped")).otherwise(F.lit("missing_actual_mapping"))))

    frames = [project("demand", "cumulative_sales", "sales", "owned", "to_sales_name_match"),
              project("demand", lease_metric_name.strip(), "lease", "leased", "to_rental_name_match")]
    for metric in metrics:
        frames.extend([project("performance", metric, "sales", "owned", "to_eilers_ts_name_match"),
                       project("performance", metric, "lease", "leased", "to_eilers_ts_name_match")])
    return reduce(lambda left, right: left.unionByName(right), frames)


def predictions_to_long(demand_df, perf_df, metrics):
    """Project saved predictions only; never read 06's actual placeholders.

    Missing prediction values stay in this frame so evaluation can report
    coverage against its independently frozen actual points.
    """
    F = _ea_functions()
    metrics = _ea_metrics(metrics)
    lineage = ["forecast_run_id", "forecast_mlflow_run_id", "forecast_config_hash", "experiment_id", "experiment_name",
               "similarity_mlflow_run_id", "similarity_config_hash"]

    def project_base(frame, label):
        _ea_require(frame, ["forecast_run_id", "target_game_key", "month_index", "revenue_type", "own_status"], label)
        frame = frame.withColumn("target_game_key", _ea_normalized("target_game_key"))
        if "_target_id" not in frame.columns:
            frame = frame.withColumn("_target_id", F.xxhash64("target_game_key"))
        if "target_game" not in frame.columns:
            frame = frame.withColumn("target_game", F.col("target_game_key"))
        for name in ["game_release_date", "prediction_date"]:
            frame = frame.withColumn(name, _ea_date(name) if name in frame.columns else F.lit(None).cast("date"))
        for name in lineage:
            if name not in frame.columns:
                frame = frame.withColumn(name, F.lit(None).cast("string"))
        if "target_curve_starts_before_prediction" not in frame.columns:
            frame = frame.withColumn("target_curve_starts_before_prediction", F.lit(False))
        return frame

    shared = [*lineage, *_EA_TARGET_COLUMNS, "revenue_type", "own_status", "month_index", "target_curve_starts_before_prediction"]
    demand = project_base(demand_df, "Saved demand predictions")
    _ea_require(demand, ["metric_name", "forecast_value", "forecast_month"], "Saved demand predictions")
    fit_columns = [column for curve, (column, _) in DEMAND_CURVES.items()
                   if curve != "baseline" and column in demand.columns]
    frames = [demand.select(*shared, F.lit("demand").alias("forecast_type"), "metric_name",
                            _ea_date("forecast_month").alias("forecast_month"), _ea_finite("forecast_value").alias("prediction"),
                            *[_ea_finite(column).alias(column) for column in fit_columns])]
    performance = project_base(perf_df, "Saved performance predictions")
    _ea_require(performance, [*[metric + "_mean" for metric in metrics]], "Saved performance predictions")
    if "forecast_month" in performance.columns:
        forecast_month = _ea_date("forecast_month")
    elif "forecast_yearmonth" in performance.columns:
        forecast_month = F.expr("try_cast(concat(substring(cast(`forecast_yearmonth` AS STRING), 1, 4), '-', "
                                "substring(cast(`forecast_yearmonth` AS STRING), 5, 2), '-01') AS DATE)")
    else:
        raise ValueError("Saved performance predictions require forecast_month or forecast_yearmonth.")
    for metric in metrics:
        frames.append(performance.select(*shared, F.lit("performance").alias("forecast_type"), F.lit(metric).alias("metric_name"),
                                         forecast_month.alias("forecast_month"), _ea_finite(metric + "_mean").alias("prediction")))
    return reduce(lambda left, right: left.unionByName(right, allowMissingColumns=True), frames)


def prepare_evaluation_actuals(spark, source_frames, target_tasks_df, settings, source_settings):
    """Load actual labels independently from explicit pinned Spark sources.

    settings supplies evaluation_months (or horizon_months). source_settings
    supplies forecast-compatible lease_value_column/lease_metric_name,
    performance_metrics and slot_counts. Sources must already be pinned by the
    caller. All labels use the target's explicit prediction month for indexing.

    Sales accumulate from the first observed source sales month, including any
    pre-prediction history. Such curves are flagged, allowing the evaluation
    policy to exclude them. Internal sales gaps contribute zero but remain
    unobserved labels. Rental actuals never fill gaps. Performance actuals use
    duplicate slot-bucket means and metric-specific positive-slot denominators.
    """
    F = _ea_functions()
    from pyspark.sql.window import Window
    if not {"sales", "lease", "performance"}.issubset(source_frames):
        raise ValueError("Evaluation actuals require pinned sales, lease, and performance source frames.")
    source_settings = source_settings.get("settings", source_settings)
    horizon = settings.get("evaluation_months", settings.get("horizon_months", 12))
    if isinstance(horizon, bool) or not isinstance(horizon, Integral) or horizon < 1:
        raise ValueError("evaluation_months must be a positive integer.")
    metrics = _ea_metrics(settings.get("performance_metrics", source_settings.get("performance_metrics", _EA_DEFAULT_METRICS)))
    lease_value = source_settings.get("lease_value_column", "source_row_count")
    lease_metric = source_settings.get("lease_metric_name", "monthly_rental_records")
    slots = list(source_settings.get("slot_counts", []))
    _ea_require(target_tasks_df, [*_EA_TARGET_COLUMNS, *_EA_TASK_KEYS, "actual_source_key", "mapping_available", "actual_mapping_status"], "Evaluation tasks")
    sales, lease, performance = (source_frames[key] for key in ["sales", "lease", "performance"])
    _ea_require(sales, ["normalized_game_name", "revenue_type", "beginning_month_date", "expr1_sum"], "Sales actual source")
    _ea_require(lease, ["theme_name_eilers", "month", lease_value], "Rental actual source")
    _ea_require(performance, ["game_name", "own_status", "yearmonth", "no_of_slots", *metrics], "Eilers actual source")
    tasks = target_tasks_df.where(F.col("mapping_available") & F.col("actual_source_key").isNotNull())

    def needed(forecast_type, revenue_type=None):
        selected = tasks.where(F.col("forecast_type") == forecast_type)
        if revenue_type is not None:
            selected = selected.where(F.col("revenue_type") == revenue_type)
        return selected.select("actual_source_key").distinct()

    sales_monthly = (
        sales.where(_ea_normalized("revenue_type").isin("sale", "sales"))
        .select(_ea_normalized("normalized_game_name").alias("actual_source_key"),
                F.trunc(_ea_date("beginning_month_date"), "month").alias("_calendar_month"), _ea_finite("expr1_sum").alias("_monthly_value"))
        .where(_ea_valid_mapping("actual_source_key") & F.col("_calendar_month").isNotNull() & F.col("_monthly_value").isNotNull())
        .join(needed("demand", "sales"), "actual_source_key", "left_semi")
        .groupBy("actual_source_key", "_calendar_month").agg(F.sum("_monthly_value").alias("_monthly_value"))
        .where(_ea_finite("_monthly_value").isNotNull())
    )
    sales_bounds = sales_monthly.groupBy("actual_source_key").agg(
        F.min("_calendar_month").alias("actual_first_observed_month"), F.max("_calendar_month").alias("actual_last_observed_month"))
    sales_window = Window.partitionBy("actual_source_key").orderBy("_calendar_month").rowsBetween(Window.unboundedPreceding, Window.currentRow)
    sales_points = (
        sales_bounds.withColumn("_calendar_month", F.explode(F.sequence("actual_first_observed_month", "actual_last_observed_month", F.expr("interval 1 month"))))
        .join(sales_monthly, ["actual_source_key", "_calendar_month"], "left")
        .withColumn("actual_is_observed", F.col("_monthly_value").isNotNull())
        .withColumn("actual_contains_imputed_history", F.max(F.col("_monthly_value").isNull().cast("int")).over(sales_window) > 0)
        .withColumn("actual", _ea_finite(F.sum(F.coalesce("_monthly_value", F.lit(0.0))).over(sales_window)))
        .withColumn("forecast_type", F.lit("demand")).withColumn("metric_name", F.lit("cumulative_sales"))
        .withColumn("revenue_type", F.lit("sales")).withColumn("own_status", F.lit("owned"))
    )

    rental_monthly = (
        lease.select(_ea_normalized("theme_name_eilers").alias("actual_source_key"),
                     F.trunc(_ea_date("month"), "month").alias("_calendar_month"), _ea_finite(lease_value).alias("actual"))
        .where(_ea_valid_mapping("actual_source_key") & F.col("_calendar_month").isNotNull() & F.col("actual").isNotNull())
        .join(needed("demand", "lease"), "actual_source_key", "left_semi")
        .groupBy("actual_source_key", "_calendar_month").agg(F.sum("actual").alias("actual"))
        .where(_ea_finite("actual").isNotNull())
    )
    rental_window = Window.partitionBy("actual_source_key")
    rental_points = (rental_monthly.withColumn("actual_first_observed_month", F.min("_calendar_month").over(rental_window))
                     .withColumn("actual_last_observed_month", F.max("_calendar_month").over(rental_window))
                     .withColumn("actual_is_observed", F.lit(True)).withColumn("actual_contains_imputed_history", F.lit(False))
                     .withColumn("forecast_type", F.lit("demand")).withColumn("metric_name", F.lit(lease_metric))
                     .withColumn("revenue_type", F.lit("lease")).withColumn("own_status", F.lit("leased")))

    performance_rows = (
        performance.select(_ea_normalized("game_name").alias("actual_source_key"), _ea_normalized("own_status").alias("own_status"),
                           F.trunc(_ea_date("yearmonth"), "month").alias("_calendar_month"),
                           _ea_finite("no_of_slots").alias("no_of_slots"), *[_ea_finite(metric).alias(metric) for metric in metrics])
        .where(_ea_valid_mapping("actual_source_key") & F.col("own_status").isin("owned", "leased")
               & F.col("_calendar_month").isNotNull() & (F.col("no_of_slots") > 0))
        .join(needed("performance"), "actual_source_key", "left_semi")
    )
    if slots:
        performance_rows = performance_rows.where(F.col("no_of_slots").isin([float(value) for value in slots]))
    bucket = performance_rows.groupBy("actual_source_key", "own_status", "_calendar_month", "no_of_slots").agg(
        *[F.avg(metric).alias(metric) for metric in metrics])
    aggregates = []
    for metric in metrics:
        weight = F.when(F.col(metric).isNotNull(), F.col("no_of_slots"))
        aggregates.append(F.when(F.sum(weight) > 0, F.sum(weight * F.col(metric)) / F.sum(weight)).alias(metric + "_actual"))
    weighted = bucket.groupBy("actual_source_key", "own_status", "_calendar_month").agg(*aggregates)
    performance_frames = []
    for metric in metrics:
        metric_window = Window.partitionBy("actual_source_key", "own_status")
        metric_points = (weighted.select("actual_source_key", "own_status", "_calendar_month", _ea_finite(metric + "_actual").alias("actual"))
                         .where(F.col("actual").isNotNull())
                         .withColumn("actual_first_observed_month", F.min("_calendar_month").over(metric_window))
                         .withColumn("actual_last_observed_month", F.max("_calendar_month").over(metric_window))
                         .withColumn("actual_is_observed", F.lit(True)).withColumn("actual_contains_imputed_history", F.lit(False))
                         .withColumn("forecast_type", F.lit("performance")).withColumn("metric_name", F.lit(metric))
                         .withColumn("revenue_type", F.when(F.col("own_status") == "owned", F.lit("sales")).otherwise(F.lit("lease"))))
        performance_frames.append(metric_points)

    point_columns = ["actual_source_key", *_EA_TASK_KEYS, "_calendar_month", "actual", "actual_is_observed",
                     "actual_contains_imputed_history", "actual_first_observed_month", "actual_last_observed_month"]
    all_source_points = reduce(lambda left, right: left.unionByName(right),
                               [frame.select(*point_columns) for frame in [sales_points, rental_points, *performance_frames]])
    points = (tasks.join(all_source_points, ["actual_source_key", *_EA_TASK_KEYS], "inner")
              .withColumn("forecast_month", F.col("_calendar_month"))
              .withColumn("actual_source_month", F.col("_calendar_month"))
              .withColumn("month_index", F.floor(F.months_between(F.col("_calendar_month"), F.trunc("prediction_date", "month"))).cast("int") + 1)
              .where(F.col("month_index").between(1, int(horizon)))
              .withColumn("target_curve_starts_before_prediction", F.coalesce(F.col("actual_first_observed_month") < F.trunc("prediction_date", "month"), F.lit(False)))
              .withColumn("actual_status", F.when(F.col("actual_is_observed"), F.lit("observed")).otherwise(F.lit("imputed"))))
    return points.drop("_calendar_month")


In [0]:
"""Candidate-independent truth cohorts and forecast scoring for notebook 07.

No Spark, source reads, tracking, or persistence. Missing ground truth is explicit
eligibility metadata; missing predictions remain in eligible coverage denominators.
"""
import copy
import hashlib
import json
import math
from datetime import date, datetime
from numbers import Integral, Real
import numpy as np
import pandas as pd

TASK_KEYS = ["forecast_type", "metric_name", "revenue_type", "own_status"]
SERIES_KEYS = [*TASK_KEYS, "target_game_key"]
POINT_KEYS = [*SERIES_KEYS, "month_index"]
DATE_COLUMNS = ["game_release_date", "prediction_date", "forecast_month"]


def validate_evaluation_settings(settings):
    values = {
        "evaluation_months": settings.get("evaluation_months", 12),
        "minimum_observed_months": settings.get("minimum_observed_months", settings.get("min_months", 2)),
        "bracket_month": settings.get("bracket_month", 12),
        "validation_fraction": settings.get("validation_fraction", 0.2),
        "test_fraction": settings.get("test_fraction", 0.2),
        "exclude_pre_release_curves": settings.get("exclude_pre_release_curves", True),
        "exclude_cumulative_imputed_history": settings.get("exclude_cumulative_imputed_history", False),
        "performance_metrics": list(settings.get("performance_metrics", [
            "avg_coin_in_index_vs_house", "avg_theo_net_win_index_vs_house"])),
    }
    for name in ["evaluation_months", "minimum_observed_months", "bracket_month"]:
        _positive_int(values[name], name)
        values[name] = int(values[name])
    if values["minimum_observed_months"] > values["evaluation_months"] or values["bracket_month"] > values["evaluation_months"]:
        raise ValueError("minimum_observed_months and bracket_month cannot exceed evaluation_months.")
    for name in ["validation_fraction", "test_fraction"]:
        value = values[name]
        if isinstance(value, bool) or not isinstance(value, Real) or not math.isfinite(float(value)) or not 0 < value < 1:
            raise ValueError(f"{name} must be finite and strictly between zero and one.")
        values[name] = float(value)
    if values["validation_fraction"] + values["test_fraction"] >= 1:
        raise ValueError("Validation and test fractions must sum to less than one.")
    for name in ["exclude_pre_release_curves", "exclude_cumulative_imputed_history"]:
        if not isinstance(values[name], (bool, np.bool_)):
            raise ValueError(f"{name} must be boolean.")
        values[name] = bool(values[name])
    if any(not isinstance(metric, str) or not metric for metric in values["performance_metrics"]):
        raise ValueError("performance_metrics must contain column names.")
    if len(values["performance_metrics"]) != len(set(values["performance_metrics"])):
        raise ValueError("performance_metrics cannot contain duplicates.")
    return values


def _frame(records):
    records = records.to_dict("records") if isinstance(records, pd.DataFrame) else [dict(row) for row in records]
    frame = pd.DataFrame(records)
    # Never allow nullable aggregate IDs to transit through float64.
    if records and any("_target_id" in row for row in records):
        frame["_target_id"] = pd.Series([row.get("_target_id") for row in records], dtype="object")
    return frame


def _long_frame(records, settings, required, label):
    frame = _frame(records)
    if "metric_name" not in frame and "metric" in frame:
        frame["metric_name"] = frame["metric"]
    if "forecast_type" not in frame and "metric_name" in frame:
        frame["forecast_type"] = np.where(frame["metric_name"].isin(settings["performance_metrics"]), "performance", "demand")
    if frame.empty:
        for column in required:
            if column not in frame:
                frame[column] = pd.Series(dtype="object")
    _required_columns(frame, required, label)
    frame = _canonical_keys(frame)
    invalid_key = frame["target_game_key"].isna() | frame["target_game_key"].isin(["", "none", "null", "nan", "na", "n/a", "<na>"])
    if invalid_key.any():
        raise ValueError(f"{label} requires valid canonical game keys.")
    for name in TASK_KEYS:
        if frame[name].isna().any() or frame[name].astype(str).str.strip().eq("").any():
            raise ValueError(f"{label} requires nonempty task keys: {name}.")
    frame["metric"] = frame["metric_name"]
    return frame


def _json_scalar(value):
    if value is None or value is pd.NA or value is pd.NaT:
        return None
    if isinstance(value, np.generic):
        value = value.item()
    if isinstance(value, (date, datetime, pd.Timestamp)):
        return value.isoformat()
    if isinstance(value, float) and not math.isfinite(value):
        return None
    return value


def cohort_records_signature(records):
    normalized = [{key: _json_scalar(value) for key, value in row.items()} for row in records]
    hashes = sorted(hashlib.sha256(json.dumps(row, sort_keys=True, separators=(",", ":"), allow_nan=False).encode()).hexdigest()
                    for row in normalized)
    return {"row_count": len(hashes), "sha256": hashlib.sha256("\n".join(hashes).encode()).hexdigest()}


def build_evaluation_cohort(actual_records, target_records, settings):
    """Freeze observed labels and canonical splits without candidate predictions.

    Target records define every game/task in the evaluation universe, including
    tasks with no actual mapping or no usable observations. Only tasks with the
    required observed history enter points; all tasks remain in eligibility.
    """
    values = validate_evaluation_settings(settings)
    target_required = [*SERIES_KEYS, "_target_id", "target_game", "game_release_date", "prediction_date"]
    targets = _long_frame(target_records, values, target_required, "Target universe")
    for column in ["game_release_date", "prediction_date"]:
        targets[column] = _dates(targets[column])
    if targets["_target_id"].isna().any():
        raise ValueError("Every canonical game requires a stable _target_id.")
    for key, group in targets.groupby("target_game_key", sort=True, dropna=False):
        for column in ["_target_id", "game_release_date", "prediction_date"]:
            if group[column].dropna().nunique() > 1:
                raise ValueError(f"Conflicting {column} for canonical game {key!r}.")
    # Input may be a monthly target scaffold. Task metadata must still agree.
    optional = [name for name in ["actual_source_key", "mapping_available", "actual_mapping_status"] if name in targets]
    target_columns = [*target_required, "metric", *optional]
    universe = targets[target_columns].drop_duplicates().copy()
    if universe.duplicated(SERIES_KEYS).any():
        raise ValueError("Target universe has conflicting task metadata.")
    actual_required = [*POINT_KEYS, "forecast_month", "actual", "actual_is_observed"]
    actuals = _long_frame(actual_records, values, actual_required, "Actual labels")
    for column in ["actual_contains_imputed_history", "target_curve_starts_before_prediction"]:
        if column not in actuals:
            actuals[column] = False
        actuals[column] = actuals[column].map(_true)
    actuals["actual"] = pd.to_numeric(actuals["actual"], errors="coerce")
    actuals["month_index"] = pd.to_numeric(actuals["month_index"], errors="coerce")
    actuals["actual_is_observed"] = actuals["actual_is_observed"].map(_true)
    actuals["forecast_month"] = _dates(actuals["forecast_month"], month=True)
    actuals["_actual_prediction_date"] = _dates(actuals["prediction_date"]) if "prediction_date" in actuals else None
    base = (actuals["actual_is_observed"] & np.isfinite(actuals["actual"])
            & actuals["month_index"].between(1, values["evaluation_months"]) & actuals["month_index"].mod(1).eq(0))
    if values["exclude_cumulative_imputed_history"]:
        base &= ~((actuals["forecast_type"] == "demand") & (actuals["revenue_type"] == "sales") & actuals["actual_contains_imputed_history"])
    labels = actuals.loc[base, [*POINT_KEYS, "forecast_month", "actual", "_actual_prediction_date",
                                "actual_contains_imputed_history", "target_curve_starts_before_prediction"]].copy()
    labels["month_index"] = labels["month_index"].astype("int64")
    if labels.duplicated(POINT_KEYS).any():
        raise ValueError("Actual labels contain duplicate observed canonical game/task/month points.")
    outside_count = int(len(labels.merge(universe[SERIES_KEYS], on=SERIES_KEYS, how="left", indicator=True).query("_merge == 'left_only'")))
    labels = labels.merge(universe, on=SERIES_KEYS, how="inner", validate="many_to_one")
    labels = labels.rename(columns={"forecast_month": "actual_forecast_month"})
    labels["_expected_forecast_month"] = [
        None if prediction is None or pd.isna(prediction) else
        (pd.Timestamp(prediction).to_period("M").to_timestamp() + pd.DateOffset(months=int(month) - 1)).date()
        for prediction, month in zip(labels["prediction_date"], labels["month_index"])
    ]
    labels["forecast_month"] = labels["actual_forecast_month"]
    prior_series = labels.groupby(SERIES_KEYS, dropna=False)["target_curve_starts_before_prediction"].transform("max")
    calendar_bad = (labels["actual_forecast_month"].isna() | labels["_expected_forecast_month"].isna()
                      | labels["actual_forecast_month"].ne(labels["_expected_forecast_month"])
                      | (labels["_actual_prediction_date"].notna() & labels["_actual_prediction_date"].ne(labels["prediction_date"])))
    label_date_bad = calendar_bad | (prior_series if values["exclude_pre_release_curves"] else False)
    labels["_invalid_actual_dates"] = label_date_bad
    labels["_usable_date"] = ~calendar_bad
    if values["exclude_pre_release_curves"]:
        labels["_usable_date"] &= ~label_date_bad
    observed_counts = labels.groupby(SERIES_KEYS, dropna=False).agg(
        raw_observed_months=("month_index", "count"), reference_observed_months=("_usable_date", "sum"),
        invalid_actual_dates=("_invalid_actual_dates", "max")).reset_index()
    eligibility = universe.merge(observed_counts, on=SERIES_KEYS, how="left", validate="one_to_one")
    eligibility["reference_observed_months"] = eligibility["reference_observed_months"].fillna(0).astype("int64")
    eligibility["observed_months"] = eligibility["reference_observed_months"]
    eligibility["raw_observed_months"] = eligibility["raw_observed_months"].fillna(0).astype("int64")
    eligibility["invalid_actual_dates"] = eligibility["invalid_actual_dates"].map(_true)
    eligibility["invalid_dates"] = eligibility["game_release_date"].isna() | eligibility["prediction_date"].isna()
    enough = eligibility["observed_months"] >= values["minimum_observed_months"]
    eligibility["invalid_dates"] |= (eligibility["invalid_actual_dates"] & ~enough
                                       & eligibility["raw_observed_months"].ge(values["minimum_observed_months"]))
    eligibility["evaluation_status"] = np.where(eligibility["invalid_dates"], "invalid_dates", np.where(enough, "eligible", "forecast_only"))
    eligibility["eligibility_reason"] = np.where(eligibility["observed_months"].eq(0), "no_observed_actuals",
                                                  np.where(~enough, "insufficient_observed_actuals",
                                                           np.where(eligibility["invalid_dates"], "invalid_dates", "eligible")))
    eligibility.loc[eligibility["evaluation_status"].eq("invalid_dates"), "eligibility_reason"] = "invalid_dates"
    eligibility["evaluation_eligibility_status"] = eligibility["evaluation_status"]
    eligibility["evaluation_eligible"] = eligibility["evaluation_status"].eq("eligible")
    eligible_keys = eligibility.loc[eligibility["evaluation_status"] == "eligible", SERIES_KEYS]
    points = labels.loc[labels["_usable_date"]].merge(eligible_keys, on=SERIES_KEYS, how="inner", validate="many_to_one")
    games = universe[["target_game_key", "_target_id", "target_game", "game_release_date", "prediction_date"]].drop_duplicates().copy()
    if games.duplicated("target_game_key").any():
        raise ValueError("Canonical game identities differ across evaluation tasks.")
    eligible_game_keys = set(eligible_keys["target_game_key"])
    split_records = assign_temporal_splits(games.loc[games["target_game_key"].isin(eligible_game_keys)].to_dict("records"),
                                          values["validation_fraction"], values["test_fraction"])
    split_columns = ["target_game_key", "split", "split_cutoff_date", "test_cutoff_date", "validation_cutoff_date", "eligible_games_in_cohort"]
    split_frame = pd.DataFrame(split_records)
    if split_frame.empty:
        split_frame = pd.DataFrame(columns=split_columns)
    games = games.merge(split_frame[split_columns], on="target_game_key", how="left", validate="one_to_one")
    games["split"] = games["split"].fillna("forecast_only")
    games["has_eligible_actuals"] = games["target_game_key"].isin(eligible_game_keys)
    games["eligible_games_in_cohort"] = pd.to_numeric(games["eligible_games_in_cohort"], errors="coerce").fillna(len(eligible_game_keys)).astype("int64")
    eligibility = eligibility.merge(games[["target_game_key", "split"]], on="target_game_key", how="left", validate="many_to_one")
    eligibility["canonical_game_split"] = eligibility["split"]
    eligibility["evaluation_split"] = np.where(eligibility["evaluation_status"].eq("eligible"), eligibility["split"], "forecast_only")
    points = points.merge(eligibility[[*SERIES_KEYS, "reference_observed_months", "split", "evaluation_split"]], on=SERIES_KEYS, how="left", validate="many_to_one")
    brackets = points.loc[points["month_index"].eq(values["bracket_month"]), [*SERIES_KEYS, "split", "actual"]].rename(columns={"actual": "bracket_actual"}).copy()
    brackets["bracket_percent_rank"] = np.nan
    brackets["actual_bracket"] = "middle_60"
    for _, group in brackets.groupby([*TASK_KEYS, "split"], sort=True, dropna=False):
        if len(group) > 1 and group["bracket_actual"].nunique() > 1:
            percentile = (group["bracket_actual"].rank(method="average") - 1) / (len(group) - 1)
            brackets.loc[group.index, "bracket_percent_rank"] = percentile
            brackets.loc[group.index[percentile <= 0.2], "actual_bracket"] = "bottom_20"
            brackets.loc[group.index[percentile >= 0.8], "actual_bracket"] = "top_20"
        else:
            brackets.loc[group.index, "bracket_percent_rank"] = 0.5
    bracket_columns = [*SERIES_KEYS, "split", "bracket_actual", "bracket_percent_rank", "actual_bracket"]
    points = points.merge(brackets[bracket_columns], on=[*SERIES_KEYS, "split"], how="left", validate="many_to_one")
    eligibility = eligibility.merge(brackets[bracket_columns], on=[*SERIES_KEYS, "split"], how="left", validate="one_to_one")
    for frame in [points, eligibility]:
        frame["actual_bracket"] = frame["actual_bracket"].fillna("not_available")
        frame["bracket_month"] = values["bracket_month"]
        frame["metric"] = frame["metric_name"]
    points = points.drop(columns=["_actual_prediction_date", "_invalid_actual_dates", "_expected_forecast_month", "_usable_date"], errors="ignore").sort_values(POINT_KEYS).reset_index(drop=True)
    eligibility = eligibility.sort_values(SERIES_KEYS).reset_index(drop=True)
    games = games.sort_values(["game_release_date", "target_game_key"], na_position="last").reset_index(drop=True)
    signatures = {name: cohort_records_signature(frame.to_dict("records")) for name, frame in [
        ("points", points), ("eligibility", eligibility), ("games", games)]}
    content_hash = hashlib.sha256(json.dumps(signatures, sort_keys=True, separators=(",", ":")).encode()).hexdigest()
    return {"points": points, "eligibility": eligibility, "games": games, "content_hash": content_hash,
            "signatures": signatures, "settings": copy.deepcopy(values),
            "diagnostics": {"target_tasks": len(eligibility), "eligible_tasks": int(eligibility["evaluation_status"].eq("eligible").sum()),
                            "expected_points": len(points), "actual_points_outside_target_universe": outside_count,
                            "eligibility_status_counts": {str(key): int(value) for key, value in eligibility["evaluation_status"].value_counts().items()}}}

def _required_columns(frame, columns, label):
    missing = sorted(set(columns) - set(frame.columns))
    if missing:
        raise ValueError(f"{label} is missing columns: {missing}")

def _true(value):
    if value is None or pd.isna(value):
        return False
    if isinstance(value, (bool, np.bool_)):
        return bool(value)
    return str(value).strip().lower() in {"true", "1", "yes", "y"}

def _positive_int(value, name):
    if isinstance(value, bool) or not isinstance(value, Integral) or value < 1:
        raise ValueError(f"{name} must be a positive integer.")

def _dates(values, month=False):
    values = pd.to_datetime(values, errors="coerce")
    if month:
        values = values.dt.to_period("M").dt.to_timestamp()
    return values.dt.date

def _canonical_keys(frame):
    frame = frame.copy()
    frame["target_game_key"] = (
        frame["target_game_key"].astype("string").str.strip().str.lower()
        .str.replace(r"\s+", " ", regex=True)
    )
    return frame

def assign_temporal_splits(records, val_fraction=0.20, test_fraction=0.20):
    """One chronological game split; equal release dates always stay together.

    Test uses the newest ceil(N*test_fraction) boundary, including date ties.
    Validation uses the newest ceil(N*val_fraction) boundary strictly before
    test. Remaining older dates are train. Small/tied populations can leave a
    split empty; no train or test observations are reused to fill validation.
    """
    for name, fraction in [("val_fraction", val_fraction), ("test_fraction", test_fraction)]:
        if isinstance(fraction, bool) or not isinstance(fraction, Real) or not math.isfinite(float(fraction)) or not 0 < fraction < 1:
            raise ValueError(f"{name} must be finite and strictly between zero and one.")
    if val_fraction + test_fraction >= 1:
        raise ValueError("Validation and test fractions must sum to less than one.")
    games = [dict(row) for row in records]
    if not games:
        return []
    keys = [row["target_game_key"] for row in games]
    if len(keys) != len(set(keys)):
        raise ValueError("Temporal splitting requires one record per canonical game.")
    for row in games:
        value = pd.to_datetime(row["game_release_date"], errors="coerce")
        if pd.isna(value):
            raise ValueError("Every split game needs a valid release date.")
        row["game_release_date"] = value.date()
    games.sort(key=lambda row: (row["game_release_date"], row["target_game_key"]))
    count = len(games)
    test_count = max(1, math.ceil(count * test_fraction))
    test_cutoff = games[-test_count]["game_release_date"]
    older = [row for row in games if row["game_release_date"] < test_cutoff]
    val_count = max(1, math.ceil(count * val_fraction))
    val_cutoff = older[-min(val_count, len(older))]["game_release_date"] if older else None
    result = []
    for row in games:
        released = row["game_release_date"]
        split = "test" if released >= test_cutoff else (
            "validation" if val_cutoff is not None and released >= val_cutoff else "train"
        )
        result.append({
            **row, "split": split,
            "split_cutoff_date": test_cutoff,
            "test_cutoff_date": test_cutoff,
            "validation_cutoff_date": val_cutoff,
            "eligible_games_in_cohort": count,
        })
    return result

def _evaluate_forecast_pairs(prediction_records, points_df, min_months=2, metadata=None):
    """Left-align one candidate to frozen labels and retain missing predictions.

    MAE/RMSE/bias pool observed month pairs, not game means. Directional
    accuracy compares only consecutive scored months within a game/task;
    unchanged values are a third direction category. Validation selection
    requires complete coverage of all expected points and games.
    """
    _positive_int(min_months, "min_months")
    _required_columns(points_df, [*POINT_KEYS, "_target_id", "target_game", "actual",
                                 "forecast_month", "prediction_date", "evaluation_split", "actual_bracket"], "Frozen cohort")
    points = points_df.copy().reset_index(drop=True)
    if points.duplicated(POINT_KEYS).any():
        raise ValueError("Frozen cohort has duplicate expected points.")
    metadata = dict(metadata or {})
    overlap = set(metadata) & set(points.columns)
    if overlap:
        raise ValueError(f"Run metadata cannot replace frozen label fields: {sorted(overlap)}")
    predictions = pd.DataFrame(prediction_records)
    prediction_columns = [*POINT_KEYS, "prediction", "forecast_month", "prediction_date"]
    if predictions.empty:
        predictions = pd.DataFrame(columns=prediction_columns)
    _required_columns(predictions, prediction_columns, "Candidate predictions")
    predictions = _canonical_keys(predictions[prediction_columns])
    predictions["month_index"] = pd.to_numeric(predictions["month_index"], errors="coerce")
    predictions = predictions.loc[predictions["month_index"].notna() & predictions["month_index"].mod(1).eq(0)].copy()
    predictions["month_index"] = predictions["month_index"].astype("int64")
    if predictions.duplicated(POINT_KEYS).any():
        raise ValueError("Candidate has duplicate canonical game/task/month predictions.")
    predictions["prediction"] = pd.to_numeric(predictions["prediction"], errors="coerce")
    predictions["forecast_month"] = _dates(predictions["forecast_month"], month=True)
    predictions["prediction_date"] = _dates(predictions["prediction_date"])
    predictions = predictions.rename(columns={
        "prediction": "_raw_prediction", "forecast_month": "_candidate_forecast_month",
        "prediction_date": "_candidate_prediction_date",
    })
    predictions["_candidate_row_present"] = True
    points = points.merge(predictions, on=POINT_KEYS, how="left", validate="one_to_one")
    calendar_match = (
        (points["forecast_month"] == points["_candidate_forecast_month"])
        & (points["prediction_date"] == points["_candidate_prediction_date"])
    )
    numeric_prediction = pd.to_numeric(points["_raw_prediction"], errors="coerce")
    points["prediction"] = numeric_prediction.where(calendar_match & np.isfinite(numeric_prediction))
    present = points["_candidate_row_present"].eq(True)
    points["prediction_missing_reason"] = np.where(
        ~present, "missing_point", np.where(~calendar_match, "calendar_or_prediction_date_mismatch",
                                            np.where(~np.isfinite(numeric_prediction), "nonfinite_prediction", ""))
    )
    points["error"] = points["prediction"] - points["actual"]
    points["absolute_error"] = points["error"].abs()
    points["squared_error"] = points["error"] ** 2
    points["direction_correct"] = np.nan
    scored = points.loc[points["prediction"].notna()].sort_values([*SERIES_KEYS, "month_index"])
    for _, group in scored.groupby(SERIES_KEYS, sort=False, dropna=False):
        consecutive = group["month_index"].diff().eq(1)
        direction = np.sign(group["prediction"].diff()).eq(np.sign(group["actual"].diff())).astype(float)
        points.loc[group.index, "direction_correct"] = direction.where(consecutive)
    for key, value in metadata.items():
        points[key] = value

    def summarize(group, result_level, bracket):
        paired = group.loc[group["prediction"].notna()]
        expected = len(group)
        paired_count = len(paired)
        expected_games = group["target_game_key"].nunique()
        game_counts = paired.groupby("target_game_key")["month_index"].count()
        games_meeting_minimum = int((game_counts >= min_months).sum())
        row = {key: group.iloc[0][key] for key in [*TASK_KEYS, "evaluation_split"]}
        per_game = result_level == "per_game"
        row.update({
            **metadata, "result_level": result_level, "bracket": bracket,
            "_target_id": int(group.iloc[0]["_target_id"]) if per_game else None,
            "target_game_key": str(group.iloc[0]["target_game_key"]) if per_game else None,
            "target_game": str(group.iloc[0]["target_game"]) if per_game else None,
            "expected_games": int(expected_games), "scored_games": int(len(game_counts)),
            "games_meeting_min_months": games_meeting_minimum,
            "expected_months": expected, "valid_months": paired_count,
            "missing_forecast_months": expected - paired_count,
            "forecast_coverage": paired_count / expected if expected else 0.0,
            "directional_comparisons": int(paired["direction_correct"].count()),
            "mae": float(paired["absolute_error"].mean()) if paired_count else None,
            "rmse": float(np.sqrt(paired["squared_error"].mean())) if paired_count else None,
            "bias": float(paired["error"].mean()) if paired_count else None,
            "directional_accuracy": float(paired["direction_correct"].mean()) if paired["direction_correct"].notna().any() else None,
            "aggregation": "pooled_observed_months",
        })
        row["eligible_for_selection"] = bool(
            result_level == "overall" and row["evaluation_split"] == "validation"
            and expected > 0 and paired_count == expected
            and games_meeting_minimum == expected_games
            and row["mae"] is not None and math.isfinite(row["mae"])
            and row["rmse"] is not None and math.isfinite(row["rmse"])
        )
        return row

    metrics = []
    for _, group in points.groupby([*TASK_KEYS, "evaluation_split"], sort=True, dropna=False):
        metrics.append(summarize(group, "overall", "all_games"))
        for bracket, subset in group.groupby("actual_bracket", sort=True, dropna=False):
            metrics.append(summarize(subset, "bracket", str(bracket)))
        for _, subset in group.groupby("target_game_key", sort=True, dropna=False):
            metrics.append(summarize(subset, "per_game", str(subset.iloc[0]["actual_bracket"])))
    metrics_df = pd.DataFrame(metrics)
    # Pandas would otherwise coerce nullable aggregate IDs to float and lose
    # precision for Spark xxhash64 IDs. Preserve Python integers explicitly.
    if not metrics_df.empty:
        metrics_df["_target_id"] = pd.Series([row["_target_id"] for row in metrics], dtype="object")
    return metrics_df, points

def _select_validation_winners(metrics_records):
    """Rank only full-coverage validation-overall MAE, then RMSE and run ID."""
    frame = metrics_records.copy() if isinstance(metrics_records, pd.DataFrame) else pd.DataFrame(metrics_records)
    required = [*TASK_KEYS, "forecast_run_id", "result_level", "evaluation_split",
                "eligible_for_selection", "forecast_coverage", "expected_months", "valid_months", "mae", "rmse"]
    _required_columns(frame, required, "Candidate metrics")
    board = frame.loc[(frame["result_level"] == "overall") & (frame["evaluation_split"] == "validation")].copy()
    if board.duplicated([*TASK_KEYS, "forecast_run_id", "curve_name"]).any():
        raise ValueError("Each candidate needs exactly one validation-overall row per task.")
    board["validation_rank"] = pd.Series(pd.NA, index=board.index, dtype="Int64")
    board["is_recommended"] = False
    board["selection_reason"] = "incomplete_validation_coverage"
    board["selection_status"] = "not_selected"
    numeric_mae = pd.to_numeric(board["mae"], errors="coerce")
    numeric_rmse = pd.to_numeric(board["rmse"], errors="coerce")
    eligible = board["eligible_for_selection"].map(_true)
    eligible &= board["forecast_coverage"].eq(1.0) & board["valid_months"].eq(board["expected_months"])
    eligible &= board["expected_months"].gt(0) & np.isfinite(numeric_mae) & np.isfinite(numeric_rmse)
    finite_metrics = np.isfinite(numeric_mae) & np.isfinite(numeric_rmse)
    board.loc[~finite_metrics, "selection_reason"] = "no_finite_validation_metrics"
    board.loc[eligible, "selection_reason"] = "ranked_by_validation_mae"
    for _, group in board.loc[eligible].groupby(TASK_KEYS, sort=True, dropna=False):
        ordered = group.sort_values(["mae", "rmse", "forecast_run_id", "curve_name"], kind="stable")
        board.loc[ordered.index, "validation_rank"] = list(range(1, len(ordered) + 1))
        board.loc[ordered.index[0], "is_recommended"] = True
        board.loc[ordered.index[0], "selection_status"] = "recommended_validation"
    board = board.sort_values([*TASK_KEYS, "validation_rank", "forecast_run_id", "curve_name"], na_position="last").reset_index(drop=True)
    winners = board.loc[board["is_recommended"]].copy().reset_index(drop=True)
    return board, winners


def evaluate_forecast(prediction_records, cohort_points, settings, metadata=None):
    """Score one candidate against frozen expected labels without dropping gaps."""
    values = validate_evaluation_settings(settings)
    points = _frame(cohort_points)
    metadata = copy.deepcopy(metadata or {})
    # Persisted cohorts may carry immutable cohort IDs. Allow identical metadata
    # to pass through, while rejecting an attempt to replace frozen label values.
    for name in set(metadata) & set(points.columns):
        expected = _json_scalar(metadata[name])
        if any(_json_scalar(value) != expected for value in points[name]):
            raise ValueError(f"Run metadata conflicts with frozen cohort field: {name}")
        points = points.drop(columns=[name])
    if "metric_name" not in points and "metric" in points:
        points["metric_name"] = points["metric"]
    if points.empty:
        metric_columns = [*TASK_KEYS, "metric", "evaluation_split", "result_level", "bracket", "_target_id", "target_game_key", "target_game",
                          "expected_games", "scored_games", "games_meeting_min_months", "expected_months", "valid_months", "missing_forecast_months",
                          "forecast_coverage", "directional_comparisons", "mae", "rmse", "bias", "directional_accuracy", "eligible_for_selection", "aggregation", *metadata]
        return {"metrics": pd.DataFrame(columns=list(dict.fromkeys(metric_columns))), "points": points.copy(),
                "coverage": pd.DataFrame(columns=[*SERIES_KEYS, "evaluation_split", "expected_months", "valid_months", "forecast_coverage"]),
                "diagnostics": {"expected_points": 0, "scored_points": 0, "prediction_missing_reasons": {}}}
    for column in DATE_COLUMNS:
        if column in points:
            points[column] = _dates(points[column], month=column == "forecast_month")
    predictions = _long_frame(prediction_records, values,
                              [*POINT_KEYS, "prediction", "forecast_month", "prediction_date"], "Candidate predictions")
    metrics, scored = _evaluate_forecast_pairs(predictions.to_dict("records"), points,
                                              values["minimum_observed_months"], metadata)
    metrics["metric"] = metrics["metric_name"]
    scored["metric"] = scored["metric_name"]
    per_game = metrics.loc[metrics["result_level"].eq("per_game")].copy()
    diagnostics = {"expected_points": len(scored), "scored_points": int(scored["prediction"].notna().sum()),
                   "prediction_missing_reasons": {str(reason): int(count) for reason, count in scored["prediction_missing_reason"].value_counts().items() if reason}}
    return {"metrics": metrics, "points": scored, "coverage": per_game, "diagnostics": diagnostics}


def evaluate_saved_curves(prediction_records, cohort_points, saved_demand_columns, settings, metadata, calibrate=False):
    """Use the same frozen demand labels for every saved curve, including missing fits."""
    predictions = _frame(prediction_records)
    points = _frame(cohort_points)
    baseline = evaluate_forecast(predictions, points, settings, {**metadata, "curve_name": "baseline"})
    metrics = [baseline["metrics"]]
    diagnostics = {**baseline["diagnostics"], "fitted_curves": {}}
    demand_points = points.loc[points["forecast_type"].eq("demand")] if not points.empty else points
    demand = predictions.loc[predictions["forecast_type"].eq("demand")] if not predictions.empty else predictions
    # Freeze the training window at the earliest held-out forecast origin.
    heldout = (points.loc[points["evaluation_split"].isin(["validation", "test"])]
               if not points.empty else points)
    cutoff = (pd.to_datetime(heldout["prediction_date"]).min().to_period("M").start_time
              if calibrate and not heldout.empty else None)
    calibration = {"available_from": cutoff.date().isoformat() if cutoff is not None else None, "offsets": {}}
    for curve, (column, _) in DEMAND_CURVES.items():
        if curve != "baseline" and column not in saved_demand_columns:
            continue
        if curve == "baseline":
            values = demand
            scored = baseline["points"]
            if not scored.empty:
                scored = scored.loc[scored["forecast_type"].eq("demand")]
        else:
            values = demand.assign(prediction=demand[column]) if not demand.empty else demand
            result = evaluate_forecast(values, demand_points, settings, {**metadata, "curve_name": curve})
            scored = result["points"]
            metrics.append(result["metrics"])
            diagnostics["fitted_curves"][curve] = result["diagnostics"]
        if not calibrate or cutoff is None or scored.empty:
            continue
        train = scored.loc[scored["evaluation_split"].eq("train")
                           & pd.to_datetime(scored["forecast_month"]).lt(cutoff)
                           & scored["prediction"].notna()].copy()
        train["residual"] = train["actual"] - train["prediction"]
        offsets = (train.groupby(["revenue_type", "month_index"], as_index=False, sort=True)
                   .agg(offset=("residual", "mean"), train_games=("target_game_key", "nunique")))
        calibration["offsets"][curve] = offsets.to_dict("records")
        adjusted = values.copy()
        if not adjusted.empty:
            adjusted["prediction"] = apply_training_bias(adjusted, calibration, curve, adjusted["prediction"])
        eval_points = demand_points.loc[demand_points["evaluation_split"].isin(["validation", "test"])]
        corrected = evaluate_forecast(adjusted, eval_points, settings, {**metadata, "curve_name": curve + "_train_bias"})
        metrics.append(corrected["metrics"])
        diagnostics["fitted_curves"][curve + "_train_bias"] = corrected["diagnostics"]
    if calibrate:
        diagnostics["training_bias"] = calibration
    return {"metrics": pd.concat(metrics, ignore_index=True), "diagnostics": diagnostics}


def select_validation_winners(metrics_records):
    """Rank complete-coverage validation metrics only; never inspect test errors."""
    frame = _frame(metrics_records)
    if "metric_name" not in frame and "metric" in frame:
        frame["metric_name"] = frame["metric"]
    if frame.empty:
        for name in [*TASK_KEYS, "forecast_run_id", "result_level", "evaluation_split", "eligible_for_selection",
                     "forecast_coverage", "expected_months", "valid_months", "mae", "rmse"]:
            if name not in frame:
                frame[name] = pd.Series(dtype="object")
    if "curve_name" not in frame:
        frame["curve_name"] = "baseline"
    return _select_validation_winners(frame)


## Durable cohorts, individual evaluations and selection snapshots


In [0]:
import json
import math
from datetime import date, datetime, timezone
from decimal import Decimal

def _now():
    return datetime.now(timezone.utc).isoformat()

def _normal(value):
    if hasattr(value, "asDict"):
        return _normal(value.asDict(recursive=True))
    if isinstance(value, dict):
        return {str(key): _normal(item) for key, item in value.items()}
    if isinstance(value, (list, tuple)):
        return [_normal(item) for item in value]
    if isinstance(value, (datetime, date)):
        return value.isoformat()
    if isinstance(value, Decimal):
        return str(value)
    if hasattr(value, "item"):
        return _normal(value.item())
    if isinstance(value, float) and not math.isfinite(value):
        raise ValueError("Nonfinite values cannot be persisted as canonical metadata/results.")
    return value

def _json(value):
    return json.dumps(_normal(value), sort_keys=True, separators=(",", ":"), allow_nan=False)

def _quote(name):
    return ".".join("`" + part.replace("`", "``") + "`" for part in name.split("."))

def _numeric_metrics(value, prefix=""):
    metrics = {}
    for key, item in value.items():
        name = f"{prefix}/{key}" if prefix else str(key)
        if isinstance(item, dict):
            metrics.update(_numeric_metrics(item, name))
        elif isinstance(item, (int, float)) and not isinstance(item, bool) and math.isfinite(float(item)):
            metrics[name] = float(item)
    return metrics

def _canonical_column(column, data_type):
    """Make nested maps deterministic before Spark serializes each row."""
    from pyspark.sql import functions as F
    from pyspark.sql.types import ArrayType, DoubleType, FloatType, MapType, StructType

    if isinstance(data_type, (FloatType, DoubleType)):
        # Distributed sums can differ in their final binary digits when Spark
        # changes reduction order. Only the integrity fingerprint rounds to
        # twelve significant digits; published forecast values are untouched.
        numeric = F.when(column == 0, F.lit(0.0)).otherwise(column.cast("double"))
        return F.when(column.isNull(), F.lit(None).cast("string")).otherwise(
            F.format_string("%.12g", numeric))
    if isinstance(data_type, MapType):
        value = F.transform(F.array_sort(F.map_keys(column)), lambda key: F.struct(
            key.alias("key"),
            _canonical_column(F.element_at(column, key), data_type.valueType).alias("value")))
        return F.when(column.isNull(), F.lit(None)).otherwise(value)
    if isinstance(data_type, ArrayType):
        return F.transform(column, lambda item: _canonical_column(item, data_type.elementType))
    if isinstance(data_type, StructType):
        value = F.struct(*[
            _canonical_column(column.getField(field.name), field.dataType).alias(field.name)
            for field in sorted(data_type.fields, key=lambda item: item.name)])
        return F.when(column.isNull(), F.lit(None)).otherwise(value)
    return column

def distributed_signature(frame, columns):
    """A bounded-memory, order-independent multiset signature.

    Float/double values use twelve significant digits in this fingerprint only;
    integer, decimal, date and identity fields remain exact. Count, SHA256
    extrema and five independent exact decimal hash-limb sums
    are computed on workers. The driver receives one aggregate row, never a
    global array of hashes or source data. This is an integrity fingerprint,
    rather than a proof that two arbitrary multisets are mathematically equal.
    """
    from pyspark.sql import functions as F

    fields = {field.name: field for field in frame.schema.fields}
    missing = set(columns) - set(fields)
    if missing:
        raise ValueError(f"Output is missing signature columns: {sorted(missing)}")
    canonical = F.struct(*[
        _canonical_column(F.col(name), fields[name].dataType).alias(name)
        for name in sorted(columns)])
    hashed = frame.select(F.sha2(F.to_json(canonical, {
        "ignoreNullFields": "false", "timeZone": "UTC",
        "dateFormat": "yyyy-MM-dd", "timestampFormat": "yyyy-MM-dd'T'HH:mm:ss.SSSXXX",
    }), 256).alias("digest"))
    offsets = (1, 16, 31, 46, 61)
    aggregate = hashed.agg(
        F.count("digest").alias("row_count"), F.min("digest").alias("hash_min"),
        F.max("digest").alias("hash_max"),
        *[F.sum(F.conv(F.substring("digest", offset, 15), 16, 10).cast("decimal(38,0)"))
          .alias(f"sum_{index}") for index, offset in enumerate(offsets)],
    ).first()
    return {
        "row_count": int(aggregate["row_count"]),
        "algorithm": "sha256-float12g-count-decimal-limb-sums-min-max-v2",
        "hash_sums": [str(aggregate[f"sum_{index}"] or 0) for index in range(len(offsets))],
        "hash_min": aggregate["hash_min"], "hash_max": aggregate["hash_max"],
    }

"""Immutable evaluation cohorts, per-forecast progress and ranking snapshots.

One manually started writer is supported. Delta completion records are the
authority; MLflow is independently repairable. No source tables are collected
and no destructive reset or cross-writer coordination is performed here.
"""


import hashlib
import json
import math
import platform
import time
import uuid
from datetime import date, datetime
from functools import reduce



EVAL_TASK_KEYS = ["forecast_type", "metric_name", "revenue_type", "own_status"]
EVAL_POINT_KEYS = [*EVAL_TASK_KEYS, "target_game_key", "month_index"]
EVAL_ELIGIBILITY_KEYS = [*EVAL_TASK_KEYS, "target_game_key"]
EVAL_TRACKING_COLUMNS = {
    "evaluation_mlflow_run_id", "forecast_mlflow_run_id", "similarity_mlflow_run_id",
}

EVAL_POINTS_SCHEMA = """
cohort_id string, cohort_config_hash string, target_game_key string,
_target_id long, target_game string, game_release_date date,
prediction_date date, forecast_month date, forecast_type string,
metric_name string, revenue_type string, own_status string, month_index long,
actual_forecast_month date, actual_contains_imputed_history boolean,
target_curve_starts_before_prediction boolean,
actual double, reference_observed_months long, split string,
evaluation_split string, actual_bracket string, bracket_actual double,
bracket_percent_rank double, bracket_month long
"""
EVAL_SPLITS_SCHEMA = """
cohort_id string, cohort_config_hash string, target_game_key string,
_target_id long, target_game string, game_release_date date, split string,
prediction_date date, split_cutoff_date date, test_cutoff_date date,
validation_cutoff_date date, eligible_games_in_cohort long, has_eligible_actuals boolean
"""
EVAL_ELIGIBILITY_SCHEMA = """
cohort_id string, cohort_config_hash string, target_game_key string,
_target_id long, target_game string, game_release_date date, prediction_date date,
forecast_type string, metric_name string, revenue_type string, own_status string,
evaluation_status string, eligibility_reason string, evaluation_eligible boolean,
evaluation_eligibility_status string, observed_months long,
reference_observed_months long, raw_observed_months long,
invalid_dates boolean, invalid_actual_dates boolean,
split string, canonical_game_split string, evaluation_split string,
actual_bracket string, bracket_actual double, bracket_percent_rank double, bracket_month long,
actual_source_key string, mapping_available boolean, actual_mapping_status string
"""
EVAL_METRICS_SCHEMA = """
curve_name string, evaluation_id string, evaluation_work_key string, evaluation_config_hash string,
evaluation_mlflow_run_id string, cohort_id string, policy_hash string, split_version string,
forecast_run_id string, forecast_mlflow_run_id string, forecast_config_hash string,
experiment_id string, experiment_name string, similarity_mlflow_run_id string,
similarity_config_hash string, forecast_type string, metric_name string,
revenue_type string, own_status string, evaluation_split string,
result_level string, bracket string, _target_id long, game_id long,
target_game_key string, target_game string, expected_games long, scored_games long,
games_meeting_min_months long, expected_months long, valid_months long,
missing_forecast_months long, directional_comparisons long,
forecast_coverage double, mae double, rmse double, bias double,
directional_accuracy double, eligible_for_selection boolean,
aggregation string, created_at_utc string
"""
EVAL_SELECTION_SCHEMA = """
curve_name string, selection_id string, selection_config_hash string, cohort_id string, policy_hash string,
forecast_type string, metric_name string, revenue_type string, own_status string,
selected_forecast_run_id string, selected_experiment_id string,
selected_forecast_mlflow_run_id string, selected_forecast_config_hash string,
selected_forecast_config_json string, forecast_run_id string, evaluation_id string,
validation_mae double, validation_rmse double, validation_bias double,
validation_forecast_coverage double, expected_validation_months long,
expected_validation_games long, validation_rank long, is_recommended boolean,
selection_status string, selection_policy string, created_at_utc string
"""
EVALUATION_RUN_SCHEMA = """
work_key string, evaluation_id string, evaluation_config_hash string,
implementation_version string, config_json string, configuration_json string,
cohort_id string, policy_hash string, forecast_run_id string, forecast_config_hash string,
forecast_mlflow_run_id string, experiment_id string, experiment_name string,
similarity_mlflow_run_id string, similarity_config_hash string,
source_versions_json string, source_paths_json string, data_revision string,
metadata_json string, status string, processing_status string,
tracking_status string, evaluation_mlflow_run_id string,
attempt_id string, attempt_count long, created_at_utc string,
started_at_utc string, updated_at_utc string, completed_at_utc string,
error_type string, error_message string, tracking_error string,
output_signatures_json string, output_locations_json string
"""
EVALUATION_ATTEMPT_SCHEMA = """
attempt_id string, work_key string, evaluation_id string, attempt_number long,
status string, started_at_utc string, completed_at_utc string,
error_type string, error_message string
"""
EVAL_COHORT_MANIFEST_SCHEMA = """
cohort_id string, cohort_config_hash string, cohort_content_hash string,
config_json string, cohort_config_json string, family_key string,
data_revision string, policy_hash string, source_versions_json string,
implementation_version string, status string, created_at_utc string,
completed_at_utc string, output_signatures_json string,
output_locations_json string
"""
EVAL_SELECTION_MANIFEST_SCHEMA = """
selection_id string, selection_config_hash string, cohort_id string,
config_json string, metadata_json string, content_hash string,
implementation_version string, status string, created_at_utc string,
completed_at_utc string, output_signatures_json string,
output_locations_json string
"""


def _evaluation_digest(value):
    return hashlib.sha256(_json(value).encode()).hexdigest()


def cohort_identity(configuration, content_signatures):
    return "cohort_" + _evaluation_digest({
        "configuration": configuration, "content": content_signatures,
    })[:32]


def selection_identity(configuration, content_signature):
    return "selection_" + _evaluation_digest({
        "configuration": configuration, "content": content_signature,
    })[:32]


def _evaluation_scalar(value, data_type):
    from pyspark.sql.types import BooleanType, DateType, DoubleType, FloatType, IntegralType, StringType

    if value is None or type(value).__name__ in {"NAType", "NaTType"}:
        return None
    if hasattr(value, "item"):
        value = value.item()
    if isinstance(value, float) and not math.isfinite(value):
        return None
    if isinstance(data_type, DateType):
        if isinstance(value, datetime):
            return value.date()
        return value if isinstance(value, date) else date.fromisoformat(str(value)[:10])
    if isinstance(data_type, IntegralType):
        if isinstance(value, float) and not value.is_integer():
            raise ValueError(f"Nonintegral value for integer evaluation field: {value}")
        return int(value)
    if isinstance(data_type, (FloatType, DoubleType)):
        return float(value)
    if isinstance(data_type, BooleanType):
        if isinstance(value, str):
            if value.lower() not in {"true", "false", "1", "0"}:
                raise ValueError(f"Invalid boolean evaluation field: {value}")
            return value.lower() in {"true", "1"}
        return bool(value)
    if isinstance(data_type, StringType):
        return _json(value) if isinstance(value, (dict, list, tuple)) else str(value)
    return value


def _evaluation_frame(spark, values, schema):
    """Typed records, with additional engine fields retained rather than lost."""
    if hasattr(values, "schema") and hasattr(values, "select"):
        if getattr(values, "isStreaming", False):
            raise ValueError("Evaluation persistence requires a batch DataFrame.")
        return values
    from pyspark.sql.types import BooleanType, DateType, DoubleType, LongType, StringType, StructField

    if hasattr(values, "to_dict"):
        values = values.to_dict("records")
    records = list(values or [])
    parsed = spark.createDataFrame([], schema=schema).schema
    known = {field.name for field in parsed.fields}
    for name in sorted({key for row in records for key in row} - known):
        sample = next((row[name] for row in records if row.get(name) is not None
                       and type(row[name]).__name__ not in {"NAType", "NaTType"}), None)
        if hasattr(sample, "item"):
            sample = sample.item()
        if isinstance(sample, bool):
            kind = BooleanType()
        elif isinstance(sample, int):
            kind = LongType()
        elif isinstance(sample, float):
            kind = DoubleType()
        elif isinstance(sample, date):
            kind = DateType()
        else:
            kind = StringType()
        parsed.add(StructField(name, kind, True))
    typed = [{field.name: _evaluation_scalar(row.get(field.name), field.dataType)
              for field in parsed.fields} for row in records]
    return spark.createDataFrame(typed, schema=parsed)


class EvaluationStore:
    """Durable fixed cohorts, independent evaluation units and ranking snapshots.

    Public tables: cohorts (observed points), splits, eligibility, metrics,
    selected, runs and attempts. cohort_runs and selection_runs are manifest
    tables; defaults are siblings of the supplied cohort/selected tables.
    """

    def __init__(self, spark, tables, mlflow_path, implementation_version):
        required = {"cohorts", "splits", "eligibility", "metrics", "selected", "runs", "attempts"}
        if required - set(tables):
            raise ValueError(f"Missing evaluation tables: {sorted(required - set(tables))}")
        self.spark, self.tables = spark, dict(tables)
        namespace = self.tables["cohorts"].rsplit(".", 1)[0]
        self.tables.setdefault("cohort_runs", namespace + ".forecast_evaluation_cohort_runs")
        self.tables.setdefault("selection_runs", namespace + ".forecast_selection_runs")
        self.mlflow_path, self.implementation_version = mlflow_path, implementation_version
        self._verified_cohorts = set()

    def initialize(self):
        for key, schema in [
            ("runs", EVALUATION_RUN_SCHEMA), ("attempts", EVALUATION_ATTEMPT_SCHEMA),
            ("cohort_runs", EVAL_COHORT_MANIFEST_SCHEMA), ("selection_runs", EVAL_SELECTION_MANIFEST_SCHEMA),
        ]:
            self.spark.sql(f"CREATE TABLE IF NOT EXISTS {_quote(self.tables[key])} ({schema}) USING DELTA")
            columns = set(self.spark.createDataFrame([], schema=schema).columns)
            missing = columns - set(self.spark.table(self.tables[key]).columns)
            if missing:
                raise ValueError(f"Incompatible {key} schema: missing {sorted(missing)}")
        return self

    def _merge(self, table, schema, key, record):
        name = "evaluation_store_" + uuid.uuid4().hex
        self.spark.createDataFrame([record], schema=schema).createOrReplaceTempView(name)
        try:
            self.spark.sql(
                f"MERGE INTO {_quote(table)} AS target USING `{name}` AS source "
                f"ON target.`{key}` = source.`{key}` "
                "WHEN MATCHED THEN UPDATE SET * WHEN NOT MATCHED THEN INSERT *")
        finally:
            self.spark.catalog.dropTempView(name)

    def _lookup(self, table, key, identity):
        from pyspark.sql import functions as F

        if not self.spark.catalog.tableExists(table):
            return None
        records = self.spark.table(table).where(F.col(key) == identity).limit(2).collect()
        if len(records) > 1:
            raise RuntimeError(f"Duplicate {key}={identity} records in {table}.")
        return records[0].asDict(recursive=True) if records else None

    def get_run(self, work_key):
        return self._lookup(self.tables["runs"], "work_key", work_key)

    def _save(self, record):
        record.update(status=record["processing_status"], updated_at_utc=_now())
        self._merge(self.tables["runs"], EVALUATION_RUN_SCHEMA, "work_key", record)
        return record

    def _attempt(self, record, status, error=None):
        self._merge(self.tables["attempts"], EVALUATION_ATTEMPT_SCHEMA, "attempt_id", {
            "attempt_id": record["attempt_id"], "work_key": record["work_key"],
            "evaluation_id": record["evaluation_id"], "attempt_number": record["attempt_count"],
            "status": status, "started_at_utc": record["started_at_utc"],
            "completed_at_utc": None if status == "running" else record.get("completed_at_utc") or _now(),
            "error_type": None if error is None else type(error).__name__,
            "error_message": None if error is None else str(error)[:16000],
        })

    def _stored(self, table, key, identity):
        from pyspark.sql import functions as F

        if not self.spark.catalog.tableExists(table):
            return None
        return self.spark.table(table).where(F.col(key) == identity)

    def _keys(self, frame, keys, label, allow_null=False):
        from pyspark.sql import functions as F

        missing = set(keys) - set(frame.columns)
        if missing:
            raise ValueError(f"{label} is missing keys: {sorted(missing)}")
        if not allow_null and frame.where(reduce(lambda a, b: a | b,
                                                 [F.col(key).isNull() for key in keys])).limit(1).count():
            raise ValueError(f"{label} contains null required identity fields.")
        if frame.groupBy(*keys).count().where(F.col("count") > 1).limit(1).count():
            raise ValueError(f"{label} contains duplicate identity fields {keys}.")

    def _signature(self, frame, keys, exclude=(), allow_null_keys=False):
        self._keys(frame, keys, "Evaluation output", allow_null_keys)
        columns = sorted(set(frame.columns) - set(exclude))
        return {**distributed_signature(frame, columns), "columns": columns,
                "key_columns": list(keys), "allow_null_keys": bool(allow_null_keys)}

    def _verify_snapshot(self, table, key, identity, expected):
        frame = self._stored(table, key, identity)
        if frame is None:
            raise RuntimeError(f"Missing saved evaluation output: {table}")
        actual = self._signature(frame.select(*expected["columns"]), expected["key_columns"],
                                 allow_null_keys=expected.get("allow_null_keys", False))
        if actual != expected:
            raise RuntimeError(f"Saved {table} content differs for {identity}.")
        return frame

    def _append_snapshot(self, table, key, identity, frame, expected, label):
        from pyspark.sql import functions as F

        existing = self._stored(table, key, identity)
        if existing is None or not existing.limit(1).count():
            (frame.write.format("delta").mode("append").option("mergeSchema", "true")
             .option("txnAppId", f"evaluation:{key}:{identity}:{label}").option("txnVersion", 0)
             .option("userMetadata", _json({key: identity, "output": label}))
             .saveAsTable(table))
        self._verify_snapshot(table, key, identity, expected)
        commit = (self.spark.sql(f"DESCRIBE HISTORY {_quote(table)}")
                  .where((F.get_json_object(F.col("userMetadata"), f"$.{key}") == identity)
                         & (F.get_json_object(F.col("userMetadata"), "$.output") == label))
                  .agg(F.min("version").alias("v")).first()["v"])
        return {"table": table, "delta_commit_version": None if commit is None else int(commit),
                "row_count": expected["row_count"]}

    def save_cohort(self, cohort_id, cohort_config, points, splits, eligibility):
        from pyspark.sql import functions as F

        configuration = _normal(dict(cohort_config))
        config_json, config_hash = _json(configuration), _evaluation_digest(configuration)
        definitions = {
            "cohorts": (points, EVAL_POINTS_SCHEMA, EVAL_POINT_KEYS),
            "splits": (splits, EVAL_SPLITS_SCHEMA, ["target_game_key"]),
            "eligibility": (eligibility, EVAL_ELIGIBILITY_SCHEMA, EVAL_ELIGIBILITY_KEYS),
        }
        prepared, content = {}, {}
        for label, (values, schema, keys) in definitions.items():
            frame = _evaluation_frame(self.spark, values, schema).drop("cohort_id", "cohort_config_hash")
            content[label] = self._signature(frame, keys)
            prepared[label] = frame
        generated_id = cohort_identity(configuration, content)
        cohort_id = cohort_id or generated_id
        content_hash = _evaluation_digest(content)
        existing = self._lookup(self.tables["cohort_runs"], "cohort_id", cohort_id)
        if existing and (existing["cohort_config_hash"] != config_hash
                         or existing["cohort_content_hash"] != content_hash):
            raise RuntimeError("This cohort ID already belongs to different immutable policy/labels/membership.")
        signatures = {}
        for label, frame in prepared.items():
            frame = (frame.withColumn("cohort_id", F.lit(cohort_id))
                     .withColumn("cohort_config_hash", F.lit(config_hash)))
            prepared[label] = frame
            signatures[label] = self._signature(frame, definitions[label][2])
        manifest = existing or {
            "cohort_id": cohort_id, "cohort_config_hash": config_hash,
            "cohort_content_hash": content_hash, "config_json": config_json,
            "cohort_config_json": config_json, "family_key": configuration.get("family_key"),
            "data_revision": str(configuration.get("data_revision", "")),
            "policy_hash": configuration.get("policy_hash"),
            "source_versions_json": _json(configuration.get("source_versions_resolved", {})),
            "implementation_version": self.implementation_version,
            "created_at_utc": _now(), "completed_at_utc": None,
            "output_locations_json": None,
        }
        if existing and json.loads(existing["output_signatures_json"]) != signatures:
            raise RuntimeError("Cohort retry produced different normalized content.")
        if existing and existing["status"] == "complete":
            for label in definitions:
                self._verify_snapshot(self.tables[label], "cohort_id", cohort_id, signatures[label])
            self._verified_cohorts.add(cohort_id)
            return existing
        manifest.update(status="running", output_signatures_json=_json(signatures))
        self._merge(self.tables["cohort_runs"], EVAL_COHORT_MANIFEST_SCHEMA, "cohort_id", manifest)
        locations = {label: self._append_snapshot(self.tables[label], "cohort_id", cohort_id,
                                                 prepared[label], signatures[label], label)
                     for label in definitions}
        manifest.update(status="complete", completed_at_utc=manifest.get("completed_at_utc") or _now(),
                        output_locations_json=_json(locations))
        self._merge(self.tables["cohort_runs"], EVAL_COHORT_MANIFEST_SCHEMA, "cohort_id", manifest)
        self._verified_cohorts.add(cohort_id)
        return manifest

    def _require_cohort(self, cohort_id):
        manifest = self._lookup(self.tables["cohort_runs"], "cohort_id", cohort_id)
        if manifest is None or manifest["status"] != "complete":
            raise ValueError("Save and complete the fixed cohort before evaluating forecasts.")
        if cohort_id not in self._verified_cohorts:
            signatures = json.loads(manifest["output_signatures_json"])
            for label in ["cohorts", "splits", "eligibility"]:
                self._verify_snapshot(self.tables[label], "cohort_id", cohort_id, signatures[label])
            self._verified_cohorts.add(cohort_id)
        return manifest

    def load_cohort(self, cohort_id):
        manifest = self._require_cohort(cohort_id)
        return {
            "manifest": manifest, "configuration": json.loads(manifest["config_json"]),
            "points_df": self._stored(self.tables["cohorts"], "cohort_id", cohort_id),
            "splits_df": self._stored(self.tables["splits"], "cohort_id", cohort_id),
            "eligibility_df": self._stored(self.tables["eligibility"], "cohort_id", cohort_id),
        }

    def _metadata(self, metadata):
        metadata = _normal(dict(metadata))
        configuration = (json.loads(metadata["config_json"]) if metadata.get("config_json") is not None
                         else metadata.get("configuration", metadata.get("config", {})))
        metadata["config_json"] = _json(configuration)
        metadata["evaluation_config_hash"] = metadata.get("evaluation_config_hash") or _evaluation_digest(configuration)
        metadata["policy_hash"] = metadata.get("policy_hash", configuration.get("policy_hash"))
        for key in ["cohort_id", "forecast_run_id"]:
            metadata[key] = metadata.get(key, configuration.get(key))
            if not metadata[key]:
                raise ValueError(f"Evaluation metadata requires {key}.")
        for key in ["forecast_config_hash", "forecast_mlflow_run_id", "experiment_id", "experiment_name",
                    "similarity_mlflow_run_id", "similarity_config_hash"]:
            metadata[key] = metadata.get(key, configuration.get(key))
        metadata["experiment_name"] = str(metadata.get("experiment_name") or metadata.get("experiment_id")
                                            or metadata["forecast_run_id"])
        metadata["source_versions"] = metadata.get("source_versions", configuration.get("source_versions_resolved", {}))
        metadata["source_paths"] = metadata.get("source_paths", configuration.get("source_paths", {}))
        metadata["data_revision"] = str(metadata.get("data_revision", configuration.get("data_revision", "")))
        return metadata

    def _assert_identity(self, record, metadata):
        for key in ["evaluation_config_hash", "config_json", "cohort_id", "policy_hash", "forecast_run_id",
                    "forecast_config_hash", "experiment_id", "similarity_config_hash", "data_revision"]:
            if record.get(key) != metadata.get(key):
                raise ValueError(f"Evaluation work key already belongs to different {key}.")
        if record["implementation_version"] != self.implementation_version:
            raise ValueError("Evaluation work key already belongs to another implementation version.")
        for key in ["source_versions", "source_paths"]:
            if record[key + "_json"] != _json(metadata[key]):
                raise ValueError(f"Evaluation work key already belongs to different {key}.")

    def begin(self, work_key, metadata):
        if not isinstance(work_key, str) or not work_key.strip():
            raise ValueError("A nonempty semantic evaluation work_key is required.")
        metadata = self._metadata({**metadata, "work_key": work_key})
        self._require_cohort(metadata["cohort_id"])
        record = self.get_run(work_key)
        if record:
            self._assert_identity(record, metadata)
            for key in ["forecast_mlflow_run_id", "similarity_mlflow_run_id"]:
                previous, incoming = record.get(key), metadata.get(key)
                if previous and incoming and previous != incoming:
                    raise ValueError(f"Conflicting upstream tracking identity: {key}")
                metadata[key] = previous or incoming
                if not previous and incoming:
                    audit = json.loads(record["metadata_json"])
                    audit[key] = incoming
                    record.update({key: incoming, "metadata_json": _json(audit), "tracking_status": "pending"})
                    self._save(record)
            if record["processing_status"] == "complete":
                self._verify_unit(record)
                self._attempt(record, "complete")
                return {**self.repair_tracking(work_key), "skip_compute": True}
            if record["processing_status"] == "running":
                self._attempt(record, "interrupted")
            metadata = {**json.loads(record["metadata_json"]), **metadata}
            metadata["requested_experiment_name"] = metadata["experiment_name"]
            metadata["experiment_name"] = record["experiment_name"]
        else:
            record = {"work_key": work_key,
                      "evaluation_id": "eval_" + hashlib.sha256(work_key.encode()).hexdigest()[:32],
                      "implementation_version": self.implementation_version,
                      "evaluation_mlflow_run_id": None, "tracking_status": "pending",
                      "attempt_count": 0, "created_at_utc": _now(), "completed_at_utc": None,
                      "output_signatures_json": None, "output_locations_json": None}
        for key in ["evaluation_config_hash", "config_json", "cohort_id", "policy_hash", "forecast_run_id", "forecast_config_hash",
                    "forecast_mlflow_run_id", "experiment_id", "experiment_name", "similarity_mlflow_run_id",
                    "similarity_config_hash", "data_revision"]:
            record[key] = metadata[key]
        record.update(configuration_json=metadata["config_json"], metadata_json=_json(metadata),
                      source_versions_json=_json(metadata["source_versions"]),
                      source_paths_json=_json(metadata["source_paths"]), processing_status="running",
                      attempt_id="attempt_" + uuid.uuid4().hex,
                      attempt_count=int(record["attempt_count"]) + 1, started_at_utc=_now(),
                      error_type=None, error_message=None, tracking_error=None)
        self._save(record)
        self._attempt(record, "running")
        return {**self._ensure_tracking_run(record), "skip_compute": False}

    def _ensure_tracking_run(self, record):
        try:
            from mlflow.tracking import MlflowClient
            from mlflow.entities import ViewType

            client = MlflowClient()
            if record.get("evaluation_mlflow_run_id"):
                run = client.get_run(record["evaluation_mlflow_run_id"])
                if run.info.lifecycle_stage != "active":
                    raise RuntimeError("The evaluation tracking run was deleted; restore it before repair.")
                return record
            experiment = client.get_experiment_by_name(self.mlflow_path)
            experiment_id = experiment.experiment_id if experiment else client.create_experiment(self.mlflow_path)
            matches = client.search_runs([experiment_id],
                filter_string=f"tags.evaluation_id = '{record['evaluation_id']}'",
                run_view_type=ViewType.ALL, max_results=2)
            if len(matches) > 1:
                raise RuntimeError("Multiple MLflow runs own this evaluation identity.")
            if matches:
                if matches[0].info.lifecycle_stage != "active":
                    raise RuntimeError("The evaluation tracking run was deleted; restore it before repair.")
                run_id = matches[0].info.run_id
            else:
                run_id = client.create_run(experiment_id, tags={
                    "mlflow.runName": f"evaluate_{record['experiment_name']}_{record['evaluation_id'][-8:]}",
                    "pipeline_stage": "evaluation", "evaluation_id": record["evaluation_id"],
                    "evaluation_work_key": record["work_key"], "cohort_id": record["cohort_id"],
                    "forecast_run_id": record["forecast_run_id"],
                    "implementation_version": self.implementation_version,
                }).info.run_id
            record.update(evaluation_mlflow_run_id=run_id, tracking_status="pending", tracking_error=None)
        except Exception as error:
            record.update(tracking_status="failed", tracking_error=f"{type(error).__name__}: {error}"[:16000])
        return self._save(record)

    def _verify_unit(self, record):
        from pyspark.sql import functions as F

        if not record.get("output_signatures_json"):
            raise RuntimeError("Evaluation completion has no expected metrics signature.")
        expected = json.loads(record["output_signatures_json"])["metrics"]
        frame = self._verify_snapshot(self.tables["metrics"], "evaluation_id", record["evaluation_id"], expected)
        for key in EVAL_TRACKING_COLUMNS:
            if frame.where(F.col(key).isNotNull()
                           & ~F.col(key).eqNullSafe(F.lit(record.get(key)))).limit(1).count():
                raise RuntimeError(f"Evaluation metrics reference a different {key}.")

    def publish(self, metrics_records, eligibility_records=None, metadata=None):
        from pyspark.sql import functions as F

        if metadata is None and isinstance(eligibility_records, dict) and "work_key" in eligibility_records:
            metadata, eligibility_records = eligibility_records, None
        if metadata is None:
            raise ValueError("publish requires metadata including work_key.")
        work_key = metadata.get("work_key")
        record = self.get_run(work_key)
        if record is None:
            raise ValueError("Call begin before evaluating/publishing a forecast.")
        old_metadata = json.loads(record["metadata_json"])
        merged = self._metadata({**old_metadata, **metadata})
        self._assert_identity(record, merged)
        merged["requested_experiment_name"] = merged["experiment_name"]
        merged["experiment_name"] = record["experiment_name"]
        cohort = self._require_cohort(record["cohort_id"])
        if eligibility_records is not None:
            incoming = _evaluation_frame(self.spark, eligibility_records, EVAL_ELIGIBILITY_SCHEMA)
            incoming = (incoming.withColumn("cohort_id", F.lit(record["cohort_id"]))
                        .withColumn("cohort_config_hash", F.lit(cohort["cohort_config_hash"])))
            expected_eligibility = json.loads(cohort["output_signatures_json"])["eligibility"]
            actual = self._signature(incoming.select(*expected_eligibility["columns"]), EVAL_ELIGIBILITY_KEYS)
            if actual != expected_eligibility:
                raise ValueError("Per-candidate eligibility cannot replace the frozen cohort eligibility.")
        if record["processing_status"] == "complete":
            self._verify_unit(record)
            return self.repair_tracking(work_key)
        if record["processing_status"] != "running":
            raise ValueError("Call begin before retrying failed evaluation work.")
        frame = _evaluation_frame(self.spark, metrics_records, EVAL_METRICS_SCHEMA)
        values = {
            "evaluation_id": record["evaluation_id"], "evaluation_work_key": work_key,
            "evaluation_config_hash": record["evaluation_config_hash"], "cohort_id": record["cohort_id"],
            "policy_hash": merged.get("policy_hash"),
            "split_version": record["cohort_id"], "created_at_utc": record["created_at_utc"],
            **{key: record.get(key) for key in ["forecast_run_id", "forecast_config_hash", "experiment_id",
               "experiment_name", "similarity_config_hash", *EVAL_TRACKING_COLUMNS]},
        }
        for key, value in values.items():
            frame = frame.withColumn(key, F.lit(value).cast("string"))
        keys = [*EVAL_TASK_KEYS, "curve_name", "evaluation_split", "result_level", "bracket", "target_game_key"]
        # Overall and bracket rows intentionally have a null canonical game.
        expected = self._signature(frame, keys, EVAL_TRACKING_COLUMNS, allow_null_keys=True)
        signatures = {"metrics": expected}
        if record.get("output_signatures_json") and json.loads(record["output_signatures_json"]) != signatures:
            raise RuntimeError("Evaluation retry produced different metrics for this immutable work key.")
        record.update(output_signatures_json=_json(signatures), metadata_json=_json(merged))
        self._save(record)
        location = self._append_snapshot(self.tables["metrics"], "evaluation_id", record["evaluation_id"],
                                         frame, expected, "metrics")
        self._verify_unit(record)
        record.update(output_locations_json=_json({"metrics": location}), processing_status="complete",
                      completed_at_utc=_now(), error_type=None, error_message=None)
        self._save(record)
        self._attempt(record, "complete")
        return self.repair_tracking(work_key)

    def persist_selection(self, selection_id, records, metadata):
        from pyspark.sql import functions as F

        metadata = _normal(dict(metadata))
        configuration = metadata.get("configuration", metadata.get("selection_config"))
        if configuration is None:
            configuration = {key: value for key, value in metadata.items() if key not in {
                "batch_id", "created_at_utc", "started_at_utc", "config_text", "requested_by"}}
        configuration = _normal(configuration)
        cohort_id = metadata.get("cohort_id", configuration.get("cohort_id"))
        self._require_cohort(cohort_id)
        frame = _evaluation_frame(self.spark, records, EVAL_SELECTION_SCHEMA)
        frame = frame.drop("selection_id", "selection_config_hash", "created_at_utc")
        frame = (frame.withColumn("cohort_id", F.lit(cohort_id))
                 .withColumn("policy_hash", F.lit(metadata.get("policy_hash", configuration.get("policy_hash"))).cast("string")))
        keys = metadata.get("selection_keys") or [*EVAL_TASK_KEYS, "curve_name", "selected_forecast_run_id", "forecast_run_id"]
        tracking = EVAL_TRACKING_COLUMNS | {"selected_forecast_mlflow_run_id"}
        content = self._signature(frame, keys, tracking, allow_null_keys=True)
        selection_id = selection_id or selection_identity(configuration, content)
        config_json, config_hash = _json(configuration), _evaluation_digest(configuration)
        content_hash = _evaluation_digest(content)
        old = self._lookup(self.tables["selection_runs"], "selection_id", selection_id)
        if old and (old["selection_config_hash"] != config_hash or old["content_hash"] != content_hash):
            raise RuntimeError("Selection snapshot ID already belongs to different candidates or results.")
        created = old["created_at_utc"] if old else _now()
        frame = (frame.withColumn("selection_id", F.lit(selection_id))
                 .withColumn("selection_config_hash", F.lit(config_hash))
                 .withColumn("created_at_utc", F.lit(created)))
        expected = self._signature(frame, keys, tracking, allow_null_keys=True)
        manifest = old or {"selection_id": selection_id, "selection_config_hash": config_hash,
                           "cohort_id": cohort_id, "config_json": config_json, "content_hash": content_hash,
                           "implementation_version": self.implementation_version,
                           "created_at_utc": created, "completed_at_utc": None,
                           "output_locations_json": None}
        signatures = {"selected": expected}
        if old and json.loads(old["output_signatures_json"]) != signatures:
            raise RuntimeError("Selection retry produced different normalized rows.")
        if old and old["status"] == "complete":
            self._verify_snapshot(self.tables["selected"], "selection_id", selection_id, expected)
            return old
        manifest.update(metadata_json=_json(metadata), status="running", output_signatures_json=_json(signatures))
        self._merge(self.tables["selection_runs"], EVAL_SELECTION_MANIFEST_SCHEMA, "selection_id", manifest)
        location = self._append_snapshot(self.tables["selected"], "selection_id", selection_id,
                                         frame, expected, "selected")
        manifest.update(status="complete", completed_at_utc=manifest.get("completed_at_utc") or _now(),
                        output_locations_json=_json({"selected": location}))
        self._merge(self.tables["selection_runs"], EVAL_SELECTION_MANIFEST_SCHEMA, "selection_id", manifest)
        return manifest

    def repair_tracking(self, work_key):
        record = self.get_run(work_key)
        if record is None:
            raise ValueError(f"Unknown evaluation work key: {work_key}")
        if record["processing_status"] != "complete" or record["tracking_status"] == "complete":
            return record
        record = self._ensure_tracking_run(record)
        if not record.get("evaluation_mlflow_run_id"):
            return record
        try:
            import mlflow
            from mlflow.tracking import MlflowClient
            from mlflow.entities import Param, Metric
            from pyspark.sql import functions as F

            self._verify_unit(record)
            client, run_id = MlflowClient(), record["evaluation_mlflow_run_id"]
            metadata, configuration = json.loads(record["metadata_json"]), json.loads(record["config_json"])
            client.log_dict(run_id, configuration, "evaluation_configuration.json")
            client.log_dict(run_id, json.loads(record["output_locations_json"]), "results/delta_outputs.json")
            cohort = self._require_cohort(record["cohort_id"])
            client.log_dict(run_id, json.loads(cohort["config_json"]), "cohort_policy.json")
            for key, path in [("coverage", "results/coverage.json"), ("diagnostics", "results/diagnostics.json")]:
                if metadata.get(key) is not None:
                    client.log_dict(run_id, metadata[key], path)
            if metadata.get("config_text") is not None:
                client.log_text(run_id, metadata["config_text"], "configuration/batch.yaml")
            client.log_dict(run_id, {"python": platform.python_version(), "spark": self.spark.version,
                "mlflow": mlflow.__version__, "implementation_version": self.implementation_version}, "environment.json")
            params = {key: record.get(key) for key in ["cohort_id", "forecast_run_id", "forecast_config_hash",
                "experiment_id", "similarity_config_hash", "data_revision", "evaluation_config_hash", "policy_hash"]}
            for source, version in json.loads(record["source_versions_json"]).items():
                params[f"source_version_{source}"] = version
            policy = configuration.get("policy", json.loads(cohort["config_json"]).get("policy", {}))
            for key, value in policy.items():
                if not isinstance(value, (dict, list)):
                    params[key] = value
            params = {key: str(value) for key, value in params.items() if value is not None}
            previous = client.get_run(run_id).data.params
            if any(previous[key] != value for key, value in params.items() if key in previous):
                raise RuntimeError("Immutable evaluation tracking parameters differ.")
            numeric = _numeric_metrics(metadata.get("coverage", {}), "coverage")
            metric_columns = [*EVAL_TASK_KEYS, "curve_name", "evaluation_split", "result_level", "bracket",
                              "mae", "rmse", "bias", "directional_accuracy", "forecast_coverage",
                              "valid_months", "expected_months", "expected_games", "scored_games",
                              "directional_comparisons"]
            # Only aggregate metric rows reach the driver for MLflow logging.
            metrics = self._stored(self.tables["metrics"], "evaluation_id", record["evaluation_id"])
            for row in metrics.where(F.col("result_level") != "per_game").select(*metric_columns).toLocalIterator():
                row = row.asDict()
                prefix = "/".join(str(row[key]) for key in ["evaluation_split", *EVAL_TASK_KEYS, "curve_name",
                                                           "result_level", "bracket"])
                for key in metric_columns[len(EVAL_TASK_KEYS) + 4:]:
                    value = row[key]
                    if value is not None and math.isfinite(float(value)):
                        numeric[f"{prefix}/{key}"] = float(value)
            timestamp = int(time.time() * 1000)
            client.log_batch(run_id, params=[Param(key, value) for key, value in params.items() if key not in previous],
                             synchronous=True)
            items = list(numeric.items())
            for offset in range(0, len(items), 500):
                client.log_batch(run_id, metrics=[Metric(key, value, timestamp, 0)
                    for key, value in items[offset:offset + 500]], synchronous=True)
            patch_name = "evaluation_tracking_" + uuid.uuid4().hex
            patch = self.spark.createDataFrame([(
                record["evaluation_id"], run_id, record.get("forecast_mlflow_run_id"),
                record.get("similarity_mlflow_run_id"))],
                "evaluation_id string, evaluation_mlflow_run_id string, forecast_mlflow_run_id string, similarity_mlflow_run_id string")
            patch.createOrReplaceTempView(patch_name)
            try:
                changed = " OR ".join(f"(target.{key} IS NULL AND source.{key} IS NOT NULL)" for key in EVAL_TRACKING_COLUMNS)
                updates = ", ".join(f"target.{key}=coalesce(target.{key}, source.{key})" for key in EVAL_TRACKING_COLUMNS)
                self.spark.sql(f"MERGE INTO {_quote(self.tables['metrics'])} AS target USING `{patch_name}` AS source "
                    f"ON target.evaluation_id=source.evaluation_id WHEN MATCHED AND ({changed}) THEN UPDATE SET {updates}")
            finally:
                self.spark.catalog.dropTempView(patch_name)
            self._verify_unit(record)
            for key in EVAL_TRACKING_COLUMNS:
                if record.get(key) is not None and self._stored(self.tables["metrics"], "evaluation_id",
                    record["evaluation_id"]).where(~F.col(key).eqNullSafe(F.lit(record[key]))).limit(1).count():
                    raise RuntimeError(f"Metrics {key} enrichment was not preserved.")
            client.set_tag(run_id, "evaluation_save_status", "complete")
            client.set_terminated(run_id, status="FINISHED")
            record.update(tracking_status="complete", tracking_error=None)
        except Exception as error:
            record.update(tracking_status="failed", tracking_error=f"{type(error).__name__}: {error}"[:16000])
        return self._save(record)

    def fail(self, work_key, error, attempt_id=None):
        record = self.get_run(work_key)
        if record is None:
            raise ValueError(f"Unknown evaluation work key: {work_key}")
        if attempt_id is not None and attempt_id != record["attempt_id"]:
            raise ValueError("Failure belongs to an earlier evaluation attempt.")
        if record["processing_status"] == "complete":
            return record
        record.update(processing_status="failed", error_type=type(error).__name__, error_message=str(error)[:16000])
        self._save(record)
        self._attempt(record, "failed", error)
        if record.get("evaluation_mlflow_run_id"):
            try:
                from mlflow.tracking import MlflowClient

                client = MlflowClient()
                client.set_tag(record["evaluation_mlflow_run_id"], "failure_type", type(error).__name__)
                client.set_terminated(record["evaluation_mlflow_run_id"],
                    status="KILLED" if isinstance(error, KeyboardInterrupt) else "FAILED")
            except Exception:
                pass
        return record

    def status_df(self):
        return self.spark.table(self.tables["runs"])


In [0]:
"""Independent benchmark registry and narrow, verified forecast loading."""


def prepare_evaluation_registry(mapping_frame, fallback_date=None):
    from pyspark.sql import functions as F
    required = {"join_game_name_key", "game_release_date"}
    if not required.issubset(mapping_frame.columns):
        raise ValueError(f"Evaluation mapping/registry snapshot lacks {sorted(required - set(mapping_frame.columns))}")
    frame = mapping_frame
    mappings = ["to_sales_name_match", "to_rental_name_match", "to_eilers_ts_name_match"]
    for name in mappings:
        if name not in frame.columns:
            frame = frame.withColumn(name, F.col("game_name") if name == "to_eilers_ts_name_match" and "game_name" in frame.columns
                                     else F.lit(None).cast("string"))
    projected = frame.select(
        _ea_normalized("join_game_name_key").alias("target_game_key"),
        F.expr("try_cast(`game_release_date` AS DATE)").alias("game_release_date"),
        *[F.when(_ea_valid_mapping(name), _ea_normalized(name)).alias(name) for name in mappings],
    ).where(_ea_valid_mapping("target_game_key"))
    conflicts = projected.groupBy("target_game_key").agg(
        *[F.countDistinct(name).alias(name) for name in ["game_release_date", *mappings]])
    bad = F.lit(False)
    for name in ["game_release_date", *mappings]:
        bad = bad | (F.col(name) > 1)
    if conflicts.where(bad).limit(1).count():
        raise ValueError("Pinned benchmark registry has conflicting release dates or source mappings for one canonical game. Resolve those mappings before refreshing the cohort.")
    registry = projected.groupBy("target_game_key").agg(
        F.min("game_release_date").alias("game_release_date"), *[F.min(name).alias(name) for name in mappings])
    return (registry.withColumn("target_game", F.col("target_game_key"))
            .withColumn("_target_id", F.xxhash64("target_game_key"))
            .withColumn("prediction_date", F.coalesce("game_release_date", F.lit(fallback_date).cast("date"))))


def load_completed_forecast_manifests(spark, config):
    from pyspark.sql import functions as F
    path = config["forecast"]["outputs"]["runs"]
    if not spark.catalog.tableExists(path):
        raise ValueError("No new-06 forecast manifest exists. Run notebook 06 first.")
    frame = spark.table(path)
    required = {"work_key", "forecast_run_id", "forecast_config_hash", "config_json", "status",
                "experiment_id", "experiment_name", "forecast_mlflow_run_id", "similarity_mlflow_run_id",
                "similarity_config_hash", "output_signatures_json", "output_locations_json"}
    if not required.issubset(frame.columns):
        raise ValueError("Forecast manifest is from the old notebook 06. Use the new incremental 06 first.")
    frame = frame.where(F.col("status") == "complete")
    selected = config["selection"]["forecast_run_ids"]
    if selected:
        available = {r["forecast_run_id"] for r in frame.select("forecast_run_id").collect()}
        if set(selected) - available:
            raise ValueError(f"Unknown/incomplete selected forecasts: {sorted(set(selected)-available)}")
        frame = frame.where(F.col("forecast_run_id").isin(selected))
    records = [row.asDict(recursive=True) for row in frame.select(*sorted(required)).collect()]
    if len({r["forecast_run_id"] for r in records}) != len(records):
        raise ValueError("Duplicate completed forecast manifests found.")
    return sorted(records, key=lambda row: row["forecast_run_id"])


def load_evaluation_prediction_frames(spark, manifest):
    """Read only one verified saved forecast; no recalculation or raw actuals."""
    import json
    from pyspark.sql import functions as F
    locations = json.loads(manifest["output_locations_json"])
    signatures = json.loads(manifest["output_signatures_json"])
    frames = {}
    try:
        for label in ["demand", "performance"]:
            frame = spark.table(locations[label]["table"]).where(F.col("forecast_run_id") == manifest["forecast_run_id"]).cache()
            frames[label] = frame
            expected = signatures[label]
            actual = distributed_signature(frame, expected["columns"])
            if any(actual[key] != expected[key] for key in actual):
                raise ValueError(f"Saved {label} output differs from its completed forecast fingerprint.")
            if frame.where(~F.col("forecast_config_hash").eqNullSafe(F.lit(manifest["forecast_config_hash"]))
                           | ~F.col("experiment_id").eqNullSafe(F.lit(manifest["experiment_id"])) ).limit(1).count():
                raise ValueError("Saved forecast lineage differs from its completed record.")
        return frames
    except BaseException:
        for frame in frames.values():
            frame.unpersist()
        raise


## Discover completed forecasts and pin label sources


In [0]:
import copy
import json
from uuid import uuid4
from datetime import datetime, timezone
from pyspark.sql import functions as F

EVALUATION_CONFIG = load_evaluation_config(CONFIG_PATH)
preflight_evaluation_tables(spark, EVALUATION_CONFIG)
EVALUATION_TABLES = EVALUATION_CONFIG["outputs"]
EVALUATION_STORE = EvaluationStore(spark, EVALUATION_TABLES,
    EVALUATION_CONFIG["tracking"]["experiment_path"], EVALUATION_IMPLEMENTATION_VERSION).initialize()
spark.sql(f"CREATE TABLE IF NOT EXISTS {quoted_table(EVALUATION_TABLES['batches'])} ({BATCH_SCHEMA}) USING DELTA")
EVALUATION_BATCH_ID = "batch_" + uuid4().hex
EVALUATION_BATCH_STARTED_AT_UTC = datetime.now(timezone.utc).isoformat()
discovery_config = {**EVALUATION_CONFIG, "selection": {"forecast_run_ids": []}}
ALL_FORECAST_MANIFESTS = load_completed_forecast_manifests(spark, discovery_config)
FORECAST_MANIFEST_BY_ID = {row["forecast_run_id"]: row for row in ALL_FORECAST_MANIFESTS}
requested_ids = EVALUATION_CONFIG["selection"]["forecast_run_ids"]
if set(requested_ids) - set(FORECAST_MANIFEST_BY_ID):
    raise ValueError(f"Unknown/incomplete selected forecasts: {sorted(set(requested_ids)-set(FORECAST_MANIFEST_BY_ID))}")
enabled_specs = {config_json(item["forecast"]) for item in EVALUATION_CONFIG["common"]["experiments"] if item["enabled"]}
shared = EVALUATION_CONFIG["forecast"]
SELECTED_FORECAST_MANIFESTS = []
for row in ALL_FORECAST_MANIFESTS:
    configuration = json.loads(row["config_json"])
    data = EVALUATION_CONFIG["common"]["data"]
    training_data = configuration.get("similarity_data", {})
    if requested_ids:
        selected = row["forecast_run_id"] in requested_ids
    else:
        selected = (config_json(configuration.get("named_experiment")) in enabled_specs
                    and training_data.get("data_revision") == data["revision"]
                    and training_data.get("source_table") == data["ai_attributes_table"]
                    and (data["source_version"] is None or training_data.get("source_table_version") == data["source_version"])
                    and (training_data.get("prediction_as_of_source") == "assumed_recent" if data["prediction_as_of_date"] is None
                         else training_data.get("prediction_as_of_source") == "configured"
                         and training_data.get("prediction_as_of_date") == data["prediction_as_of_date"])
                    and configuration.get("forecast_data_revision") == shared["data_revision"]
                    and configuration.get("source_paths") == shared["sources"]
                    and all(configuration.get(k) == v for k, v in shared["settings"].items() if k != "lease_curve_label")
                    and (not shared["selection"]["similarity_experiment_ids"]
                         or row["experiment_id"] in shared["selection"]["similarity_experiment_ids"]))
    if selected:
        SELECTED_FORECAST_MANIFESTS.append(row)
SELECTED_FORECAST_IDS = {row["forecast_run_id"] for row in SELECTED_FORECAST_MANIFESTS}
LABEL_SOURCE_VERSIONS = resolve_evaluation_source_versions(spark, EVALUATION_CONFIG) if SELECTED_FORECAST_MANIFESTS else {}
EVALUATION_POLICY = evaluation_policy(EVALUATION_CONFIG)
EVALUATION_POLICY_HASH = config_sha256(EVALUATION_POLICY)
EVALUATION_FAMILIES = {}
for row in SELECTED_FORECAST_MANIFESTS:
    family = evaluation_family(json.loads(row["config_json"]), EVALUATION_CONFIG)
    family_key = config_sha256(family)
    EVALUATION_FAMILIES.setdefault(family_key, {"configuration": family, "forecasts": []})["forecasts"].append(row)
print(f"Evaluation batch: {EVALUATION_BATCH_ID}")
print(f"Label revision: {EVALUATION_CONFIG['data_revision']}; pinned actuals/mappings versions: {LABEL_SOURCE_VERSIONS}")
print(f"Completed forecasts selected: {len(SELECTED_FORECAST_MANIFESTS)}; definition/population families: {len(EVALUATION_FAMILIES)}")
print("Only missing per-forecast evaluations are calculated. Existing cohort labels/splits are reused.")


## Evaluate missing units and refresh validation rankings


In [0]:
import json
import platform
import time
import pandas as pd
from datetime import datetime, timezone
from pyspark.sql import functions as F

if load_evaluation_config(CONFIG_PATH) != EVALUATION_CONFIG:
    raise ValueError("The YAML changed after loading. Rerun notebook 07 from the top.")

EVALUATION_BATCH_OUTCOMES = []
EVALUATION_SELECTION_SUMMARIES = []
EVALUATION_BATCH_ERRORS = []
evaluation_source_frames = None


def save_evaluation_batch(status):
    evaluation_merge_record(spark, EVALUATION_TABLES["batches"], BATCH_SCHEMA, "batch_id", {
        "batch_id": EVALUATION_BATCH_ID, "data_revision": EVALUATION_CONFIG["data_revision"],
        "status": status, "started_at_utc": EVALUATION_BATCH_STARTED_AT_UTC,
        "updated_at_utc": datetime.now(timezone.utc).isoformat(),
        "summary_json": config_json({"units": EVALUATION_BATCH_OUTCOMES, "selections": EVALUATION_SELECTION_SUMMARIES}),
        "errors_json": config_json(EVALUATION_BATCH_ERRORS)})


def evaluation_outcome(forecast, record=None, action="failed", error=None):
    record = record or {}
    metadata = json.loads(record.get("metadata_json") or "{}")
    diagnostics = metadata.get("diagnostics", {})
    return {"forecast_run_id": forecast["forecast_run_id"], "experiment_name": forecast["experiment_name"],
            "evaluation_id": record.get("evaluation_id"), "cohort_id": record.get("cohort_id"),
            "action": action, "processing_status": record.get("processing_status", "failed"),
            "tracking_status": record.get("tracking_status", "pending"),
            "expected_points": diagnostics.get("expected_points"), "scored_points": diagnostics.get("scored_points"),
            "error": None if error is None else f"{type(error).__name__}: {error}"}


def cached_label_sources():
    """Project/cache raw inputs once, only when a new cohort needs actuals."""
    frames = {}
    try:
        for key, path in EVALUATION_CONFIG["sources"].items():
            frame = spark.read.option("versionAsOf", LABEL_SOURCE_VERSIONS[key]).table(path)
            if key == "sales":
                columns = ["normalized_game_name", "revenue_type", "beginning_month_date", "expr1_sum"]
            elif key == "lease":
                quantity_columns = {family["configuration"]["lease_value_column"] for family in EVALUATION_FAMILIES.values()}
                columns = ["theme_name_eilers", "month", *sorted(quantity_columns)]
            elif key == "performance":
                columns = ["game_name", "own_status", "yearmonth", "no_of_slots", *EVALUATION_CONFIG["settings"]["performance_metrics"]]
            else:
                columns = [name for name in ["game_name", "join_game_name_key", "game_release_date", "to_sales_name_match",
                                            "to_rental_name_match", "to_eilers_ts_name_match"] if name in frame.columns]
            frames[key] = frame.select(*columns).cache()
        return frames
    except BaseException:
        for frame in frames.values():
            frame.unpersist()
        raise


def reuse_cohort(family_key):
    matches = (spark.table(EVALUATION_TABLES["cohort_runs"])
               .where((F.col("status") == "complete") & (F.col("family_key") == family_key)
                      & (F.col("data_revision") == EVALUATION_CONFIG["data_revision"])
                      & (F.col("policy_hash") == EVALUATION_POLICY_HASH)).limit(2).collect())
    if len(matches) > 1:
        raise ValueError("Multiple completed cohorts own the same family/revision/policy; resolve history rather than choosing the latest.")
    return EVALUATION_STORE.load_cohort(matches[0]["cohort_id"]) if matches else None


def save_validation_ranking(cohort_id):
    complete = spark.table(EVALUATION_TABLES["runs"]).where(
        (F.col("status") == "complete") & (F.col("cohort_id") == cohort_id)
        & (F.col("policy_hash") == EVALUATION_POLICY_HASH))
    complete_records = [r.asDict() for r in complete.select("evaluation_id", "forecast_run_id").collect()
                        if r["forecast_run_id"] in SELECTED_FORECAST_IDS]
    if not complete_records:
        return None
    ids = sorted(r["evaluation_id"] for r in complete_records)
    metric_records = [r.asDict(recursive=True) for r in spark.table(EVALUATION_TABLES["metrics"]).where(
        F.col("evaluation_id").isin(ids) & (F.col("cohort_id") == cohort_id)
        & (F.col("policy_hash") == EVALUATION_POLICY_HASH) & (F.col("result_level") == "overall")
        & (F.col("evaluation_split") == "validation")).toLocalIterator()]
    board, winners = select_validation_winners(metric_records)
    records = []
    for row in board.to_dict("records"):
        forecast = FORECAST_MANIFEST_BY_ID[row["forecast_run_id"]]
        row.update(policy_hash=EVALUATION_POLICY_HASH, cohort_id=cohort_id,
                   selected_forecast_run_id=forecast["forecast_run_id"], selected_experiment_id=forecast["experiment_id"],
                   selected_forecast_mlflow_run_id=forecast["forecast_mlflow_run_id"],
                   selected_forecast_config_hash=forecast["forecast_config_hash"], selected_forecast_config_json=forecast["config_json"],
                   validation_mae=row.get("mae"), validation_rmse=row.get("rmse"), validation_bias=row.get("bias"),
                   validation_forecast_coverage=row.get("forecast_coverage"),
                   expected_validation_months=row.get("expected_months"), expected_validation_games=row.get("expected_games"),
                   selection_policy="validation MAE, RMSE, forecast run ID and curve name; full expected coverage required")
        records.append(row)
    selection = EVALUATION_STORE.persist_selection(None, records, {
        "configuration": {"cohort_id": cohort_id, "policy_hash": EVALUATION_POLICY_HASH,
                          "evaluation_ids": ids, "selection_rule": EVALUATION_POLICY["selection_rule"]},
        "cohort_id": cohort_id, "policy_hash": EVALUATION_POLICY_HASH})
    return {"cohort_id": cohort_id, "selection_id": selection["selection_id"],
            "candidate_task_rows": len(board), "recommended_tasks": len(winners)}


save_evaluation_batch("running")
try:
    for family_key, family in EVALUATION_FAMILIES.items():
        try:
            cohort = reuse_cohort(family_key)
            if cohort is None:
                if evaluation_source_frames is None:
                    evaluation_source_frames = cached_label_sources()
                registry = prepare_evaluation_registry(evaluation_source_frames["mappings"],
                    EVALUATION_POLICY["missing_release_prediction_date"])
                family_settings = family["configuration"]
                task_frame = targets_to_tasks(registry, EVALUATION_CONFIG["settings"]["performance_metrics"],
                    lease_metric_name=family_settings["lease_metric_name"])
                actual_frame = prepare_evaluation_actuals(spark, evaluation_source_frames, task_frame,
                    EVALUATION_CONFIG["settings"], family_settings)
                # Only the narrow benchmark labels/tasks enter the pandas engine.
                target_records = [r.asDict(recursive=True) for r in task_frame.toLocalIterator()]
                actual_records = [r.asDict(recursive=True) for r in actual_frame.toLocalIterator()]
                built = build_evaluation_cohort(actual_records, target_records, EVALUATION_CONFIG["settings"])
                cohort_config = {"family_key": family_key, "data_revision": EVALUATION_CONFIG["data_revision"],
                    "policy_hash": EVALUATION_POLICY_HASH, "source_paths": EVALUATION_CONFIG["sources"],
                    "source_versions_resolved": LABEL_SOURCE_VERSIONS, "source_settings": family_settings,
                    "policy": EVALUATION_POLICY, "implementation_version": EVALUATION_IMPLEMENTATION_VERSION}
                manifest = EVALUATION_STORE.save_cohort(None, cohort_config, built["points"], built["games"], built["eligibility"])
                cohort = EVALUATION_STORE.load_cohort(manifest["cohort_id"])
                print(f"Created fixed cohort {manifest['cohort_id']}: {built['diagnostics']}")
            cohort_id = cohort["manifest"]["cohort_id"]
            cohort_points = pd.DataFrame([r.asDict(recursive=True) for r in cohort["points_df"].toLocalIterator()])
            for forecast in family["forecasts"]:
                frames = {}
                record = None
                try:
                    work_key, semantic = evaluation_work_identity(forecast, cohort_id, EVALUATION_POLICY_HASH)
                    metadata = {"work_key": work_key, "configuration": semantic, "evaluation_config_hash": work_key,
                        "cohort_id": cohort_id, "policy_hash": EVALUATION_POLICY_HASH,
                        "forecast_run_id": forecast["forecast_run_id"], "forecast_config_hash": forecast["forecast_config_hash"],
                        "forecast_mlflow_run_id": forecast["forecast_mlflow_run_id"], "experiment_id": forecast["experiment_id"],
                        "experiment_name": forecast["experiment_name"], "similarity_mlflow_run_id": forecast["similarity_mlflow_run_id"],
                        "similarity_config_hash": forecast["similarity_config_hash"], "source_versions": LABEL_SOURCE_VERSIONS,
                        "source_paths": EVALUATION_CONFIG["sources"], "data_revision": EVALUATION_CONFIG["data_revision"],
                        "forecast_configuration": json.loads(forecast["config_json"]), "batch_id": EVALUATION_BATCH_ID,
                        "config_text": EVALUATION_CONFIG["common"]["config_text"],
                        "environment": {"python": platform.python_version(), "spark": spark.version}}
                    record = EVALUATION_STORE.begin(work_key, metadata)
                    if record["skip_compute"]:
                        EVALUATION_BATCH_OUTCOMES.append(evaluation_outcome(forecast, record, "skipped_complete"))
                        continue
                    frames = load_evaluation_prediction_frames(spark, forecast)
                    for metric in EVALUATION_CONFIG["settings"]["performance_metrics"]:
                        if metric + "_mean" not in frames["performance"].columns:
                            frames["performance"] = frames["performance"].withColumn(metric + "_mean", F.lit(None).cast("double"))
                    predictions = predictions_to_long(frames["demand"], frames["performance"],
                        EVALUATION_CONFIG["settings"]["performance_metrics"]).where(
                            F.col("month_index").between(1, EVALUATION_CONFIG["settings"]["evaluation_months"]))
                    prediction_records = [r.asDict(recursive=True) for r in predictions.toLocalIterator()]
                    engine_metadata = {key: metadata[key] for key in ["cohort_id", "policy_hash", "forecast_run_id", "forecast_config_hash",
                        "forecast_mlflow_run_id", "experiment_id", "experiment_name", "similarity_mlflow_run_id", "similarity_config_hash"]}
                    engine_metadata.update(evaluation_id=record["evaluation_id"], evaluation_config_hash=work_key, split_version=cohort_id)
                    saved_columns = json.loads(forecast["output_signatures_json"])["demand"]["columns"]
                    scored = evaluate_saved_curves(prediction_records, cohort_points, saved_columns,
                                                  EVALUATION_CONFIG["settings"], engine_metadata,
                                                  calibrate=metadata["forecast_configuration"].get(
                                                      "named_experiment", {}).get("train_bias_correction", False))
                    outside = predictions.select("target_game_key").distinct().join(
                        cohort["splits_df"].select("target_game_key").distinct(), "target_game_key", "left_anti").count()
                    metadata.update(diagnostics={**scored["diagnostics"], "targets_outside_frozen_cohort": outside},
                                    coverage={key: scored["diagnostics"][key] for key in ["expected_points", "scored_points"]})
                    complete = EVALUATION_STORE.publish(scored["metrics"], metadata=metadata)
                    EVALUATION_BATCH_OUTCOMES.append(evaluation_outcome(forecast, complete, "processed"))
                except BaseException as error:
                    if record is not None:
                        try:
                            record = EVALUATION_STORE.fail(work_key, error, record["attempt_id"])
                        except Exception as status_error:
                            EVALUATION_BATCH_ERRORS.append(f"Could not persist evaluation failure: {type(status_error).__name__}: {status_error}")
                    EVALUATION_BATCH_OUTCOMES.append(evaluation_outcome(forecast, record, "failed", error))
                    EVALUATION_BATCH_ERRORS.append(f"{forecast['forecast_run_id']}: {type(error).__name__}: {error}")
                    if not isinstance(error, Exception):
                        raise
                finally:
                    for frame in frames.values():
                        frame.unpersist()
                save_evaluation_batch("running")
            selection = save_validation_ranking(cohort_id)
            if selection:
                EVALUATION_SELECTION_SUMMARIES.append(selection)
        except BaseException as error:
            EVALUATION_BATCH_ERRORS.append(f"Family {family_key}: {type(error).__name__}: {error}")
            if not isinstance(error, Exception):
                raise
finally:
    if evaluation_source_frames is not None:
        for frame in evaluation_source_frames.values():
            frame.unpersist()
    save_evaluation_batch("failed" if EVALUATION_BATCH_ERRORS else "complete")

evaluation_batch_summary_pdf = pd.DataFrame(EVALUATION_BATCH_OUTCOMES, columns=[
    "forecast_run_id", "experiment_name", "evaluation_id", "cohort_id", "action", "processing_status", "tracking_status",
    "expected_points", "scored_points", "error"])
display(evaluation_batch_summary_pdf)
display(pd.DataFrame(EVALUATION_SELECTION_SUMMARIES, columns=["cohort_id", "selection_id", "candidate_task_rows", "recommended_tasks"]))
print(f"Selected forecasts={len(SELECTED_FORECAST_MANIFESTS)}, processed={sum(r['action']=='processed' for r in EVALUATION_BATCH_OUTCOMES)}, "
      f"skipped={sum(r['action']=='skipped_complete' for r in EVALUATION_BATCH_OUTCOMES)}, failures={len(EVALUATION_BATCH_ERRORS)}")
if EVALUATION_BATCH_ERRORS:
    raise RuntimeError("Evaluation batch has failures; completed units were preserved.\n" + "\n".join(EVALUATION_BATCH_ERRORS))
if not EVALUATION_BATCH_OUTCOMES:
    print("Nothing new to evaluate.")


## Accuracy, inference eligibility and processing monitoring

Read-only saved results; game charts are in Review_Forecasts.


In [0]:
# Read-only evaluation status, label eligibility and saved metric summaries.
from pyspark.sql import functions as F

evaluation_status_df = EVALUATION_STORE.status_df().withColumn("duration_seconds",
    F.coalesce(F.col("completed_at_utc").cast("timestamp"),
        F.when(F.col("processing_status") == "running", F.current_timestamp())
         .otherwise(F.col("updated_at_utc").cast("timestamp"))).cast("double")
    - F.col("started_at_utc").cast("timestamp").cast("double"))
display(evaluation_status_df.select(
    "experiment_name", "forecast_run_id", "evaluation_id", "cohort_id", "data_revision",
    "processing_status", "tracking_status", "attempt_count", "duration_seconds",
    F.get_json_object("metadata_json", "$.diagnostics.expected_points").cast("long").alias("expected_points"),
    F.get_json_object("metadata_json", "$.diagnostics.scored_points").cast("long").alias("scored_points"),
    F.get_json_object("metadata_json", "$.diagnostics.targets_outside_frozen_cohort").cast("long").alias("targets_outside_cohort"),
    "error_message", "tracking_error", "evaluation_mlflow_run_id", "updated_at_utc",
).orderBy(F.desc("updated_at_utc")).limit(EVALUATION_CONFIG["monitoring"]["history_rows"]))
display(EVALUATION_STORE.status_df().groupBy("processing_status", "tracking_status").count())
display(spark.table(EVALUATION_TABLES["batches"]).orderBy(F.desc("updated_at_utc")).limit(
    EVALUATION_CONFIG["monitoring"]["history_rows"]))
display(spark.table(EVALUATION_TABLES["attempts"]).orderBy(F.desc("started_at_utc")).limit(
    EVALUATION_CONFIG["monitoring"]["history_rows"]))
display(spark.table(EVALUATION_TABLES["cohort_runs"]).select(
    "cohort_id", "data_revision", "family_key", "status", "source_versions_json", "created_at_utc", "completed_at_utc"
).orderBy(F.desc("created_at_utc")).limit(EVALUATION_CONFIG["monitoring"]["history_rows"]))

complete_cohorts = spark.table(EVALUATION_TABLES["cohort_runs"]).where(F.col("status") == "complete").select("cohort_id")
if spark.catalog.tableExists(EVALUATION_TABLES["eligibility"]):
    display(spark.table(EVALUATION_TABLES["eligibility"]).join(complete_cohorts, "cohort_id", "inner")
        .groupBy("cohort_id", "forecast_type", "metric_name", "own_status", "evaluation_status", "evaluation_split", "eligibility_reason").count())
if spark.catalog.tableExists(EVALUATION_TABLES["metrics"]):
    completed_units = EVALUATION_STORE.status_df().where(F.col("status") == "complete").select("evaluation_id")
    forecast_error_metrics_df = spark.table(EVALUATION_TABLES["metrics"]).join(completed_units, "evaluation_id", "inner")
    forecast_error_metrics_df = with_curve_name(forecast_error_metrics_df)
    display(forecast_error_metrics_df.where(F.col("result_level") == "overall").select(
        "experiment_name", "forecast_run_id", "cohort_id", "forecast_type", "metric_name", "curve_name", "own_status", "evaluation_split",
        "expected_games", "valid_months", "expected_months", "forecast_coverage", "mae", "rmse", "bias",
        "directional_accuracy", "eligible_for_selection", "created_at_utc",
    ).orderBy(F.desc("created_at_utc")).limit(EVALUATION_CONFIG["monitoring"]["history_rows"]))
    print("forecast_error_metrics_df includes per-game, overall and actual-bracket rows for saved train/validation/test evaluations.")
if spark.catalog.tableExists(EVALUATION_TABLES["selected"]):
    selections = spark.table(EVALUATION_TABLES["selection_runs"]).where(F.col("status") == "complete").select("selection_id")
    display(with_curve_name(spark.table(EVALUATION_TABLES["selected"])).join(selections, "selection_id", "inner")
        .where(F.col("is_recommended")).select(
            "selection_id", "cohort_id", "forecast_type", "metric_name", "own_status", "curve_name", "selected_forecast_run_id",
            "validation_mae", "validation_rmse", "validation_forecast_coverage", "created_at_utc",
        ).orderBy(F.desc("created_at_utc")).limit(EVALUATION_CONFIG["monitoring"]["history_rows"]))
print("Use Review_Forecasts for experiment/game charts and forecast-only views. No scheduling is configured.")
